
# DS5020 – Mini Project 02
## Will This Be a Strong Sales Week?

**Topic:** Bayes' Theorem and Naive Bayes on Walmart Sales Data

**Dataset:** Walmart Weekly Sales Dataset

### Objective

The objective of this project is to analyze Walmart's weekly sales data using probability, conditional probability, Bayes' theorem, and Naive Bayes classification.

We will examine how holidays, seasons, temperature, fuel prices, unemployment, and store size relate to strong sales weeks.
"


## Event Definitions (R1)

Each row represents one store in one week (a store-week).

The sample space contains 6,435 store-weeks.

| Event | Definition |
|---|---|
| S | Strong week: weekly sales at least 10% above that store's average weekly sales |
| H | Holiday week: Holiday_Flag = 1 |
| Q | Holiday season: November or December |
| C | Cold week: temperature below 40°F |
| F | Expensive fuel: fuel price at least $3.50 |
| J | High unemployment: unemployment at least 8% |
| G | Large store: average weekly sales at least $1,000,000 |

For Parts P and B, store averages are calculated using the complete dataset.

For Part N, store averages are calculated from the 2010–2011 training weeks only. The 2012 data is reserved for testing.

Every probability will be verified using counting, complements, or the sum-to-one rule (R5).


In [99]:
# Part P1 Load and Verify the Dataset

import numpy as np
import pandas as pd
df = pd.read_csv("data/Walmart.csv")
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
print("Dataset shape:", df.shape)
print("Total store-weeks:", len(df))
print("Number of stores:", df["Store"].nunique())
print("Number of weeks:", df["Date"].nunique())
print("Holiday store-weeks:", df["Holiday_Flag"].sum())
print("Missing values:", df.isna().sum().sum())
display(df.head())

Dataset shape: (6435, 8)
Total store-weeks: 6435
Number of stores: 45
Number of weeks: 143
Holiday store-weeks: 450
Missing values: 0


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106


In [100]:
#P1 Automatic Data Checks

assert df.shape == (6435, 8)
assert df["Store"].nunique() == 45
assert df["Date"].nunique() == 143
assert df["Holiday_Flag"].sum() == 450
assert df.isna().sum().sum() == 0
print("P1 Verification Passed! All dataset checks are correct.")

P1 Verification Passed! All dataset checks are correct.


### P1 - Data Verification Results

The Walmart dataset contains 6,435 store-weeks observation from 45 stores across 143 weeks. It contains 8 columns and 450 holiday store-weeks, with no missing values.

Each observation represents one store in one week. All dataset checks match the values specified in the assignment


## Part P2 – Probability of Each Event

In this section, we calculate the probability of each of the seven events: S, H, Q, C, F, J, and G.

Each probability is calculated by dividing the number of store-weeks satisfying the event by the total number of store-weeks (6,435).

The formula is:

$$
P(E)=\frac{\text{Number of store-weeks satisfying }E}{6435}
$$

According to Rule R5, every probability is verified using its complement:

$$
P(E)+P(E^c)=1
$$


In [101]:
#P2-Create Event Columns (R1)
# Calculate each Store's average weekly sales
store_avg = df.groupby("Store")["Weekly_Sales"].mean()
def add_events(d, store_avg):
    d = d.copy()
    avg = d["Store"].map(store_avg)
    d["S"] = d["Weekly_Sales"] >= 1.10 * avg
    d["H"] = d["Holiday_Flag"] == 1
    d["Q"] = d["Date"].dt.month.isin([11, 12])
    d["C"] = d["Temperature"] < 40
    d["F"] = d["Fuel_Price"] >= 3.50
    d["J"] = d["Unemployment"] >= 8
    d["G"] = avg >= 1_000_000
    return d

#Appy event definitions to the full dataset
ev = add_events(df, store_avg)
display(ev[["Store", "Date", "S", "H", "Q", "C", "F", "J", "G"]].head())

,Store,Date,S,H,Q,C,F,J,G
0,1,2010-02-05,False,False,False,False,False,True,True
1,1,2010-02-12,False,True,False,True,False,True,True
2,1,2010-02-19,False,False,False,True,False,True,True
3,1,2010-02-26,False,False,False,False,False,True,True
4,1,2010-03-05,False,False,False,False,False,True,True


In [102]:

# P2 – Probability of Each Event and Complement Verification (R5)

events = ["S", "H", "Q", "C", "F", "J", "G"]
N = len(ev)

results = []

for event in events:
    event_count = int(ev[event].sum())
    complement_count = N - event_count

    probability = event_count / N
    complement_probability = complement_count / N

    verification = probability + complement_probability

    assert np.isclose(verification, 1.0)

    results.append({
        "Event": event,
        "Count": event_count,
        "P(E)": round(probability, 6),
        "P(E^c)": round(complement_probability, 6),
        "Check": round(verification, 6)
    })

p2_table = pd.DataFrame(results)

display(p2_table)

print("P2 Verification Passed! All probabilities sum to 1 with their complements.")


,Event,Count,P(E),P(E^c),Check
0,S,814,0.126496,0.873504,1.0
1,H,450,0.069930,0.930070,1.0
2,Q,810,0.125874,0.874126,1.0
3,C,989,0.153691,0.846309,1.0
4,F,3029,0.470707,0.529293,1.0
5,J,3022,0.469619,0.530381,1.0
6,G,3146,0.488889,0.511111,1.0


P2 Verification Passed! All probabilities sum to 1 with their complements.



### P2 – Mathematical Calculation and Verification

For any event E:

$$
P(E)=\frac{\#E}{6435}
$$

**Example: Probability of a Holiday Week (H)**

There are 450 holiday store-weeks out of 6,435 observations.

$$
P(H)=\frac{450}{6435}=0.06993
$$

Therefore, approximately 6.99% of the store-weeks are holiday weeks.

**Complement Verification (R5)**

The number of non-holiday store-weeks is:

$$
6435-450=5985
$$

$$
P(H^c)=\frac{5985}{6435}=0.93007
$$

$$
P(H)+P(H^c)=0.06993+0.93007=1
$$

The same complement verification is performed for all seven events in the Python code above.



### P2 – Interpretation

The event probabilities describe how frequently strong sales weeks, holiday weeks, November–December weeks, cold weeks, expensive fuel, high unemployment, and large stores occur in the Walmart dataset.

Each probability is calculated using all 6,435 store-week observations.

All seven probabilities are independently checked using their complements, satisfying Rule R5.



## Part P3 – Joint Probabilities

A joint probability represents the probability that two events occur together.

For two events A and B, the joint probability is:

$$
P(A \cap B)=\frac{\text{Number of store-weeks where both A and B occur}}{\text{Total number of store-weeks}}
$$

Since the dataset contains 6,435 store-weeks:

$$
P(A \cap B)=\frac{\#(A \cap B)}{6435}
$$

We calculate four joint probabilities:

1. **P(S ∩ H):** Strong week and holiday week.
2. **P(S ∩ Q):** Strong week and November–December.
3. **P(H ∩ Q):** Holiday week and November–December.
4. **P(S ∩ C):** Strong week and cold week.

Each probability will be verified using its complement and counting, following Rule R5.


In [103]:

# P3 – Calculate Joint Probabilities

joint_events = [
    ("S", "H"),
    ("S", "Q"),
    ("H", "Q"),
    ("S", "C")
]

N = len(ev)

joint_results = []

for A, B in joint_events:

    # Count store-weeks where both events occur
    joint_count = int((ev[A] & ev[B]).sum())

    # Calculate joint probability
    joint_probability = joint_count / N

    joint_results.append({
        "Joint Event": f"P({A} ∩ {B})",
        "Count": joint_count,
        "Probability": round(joint_probability, 6)
    })

p3_table = pd.DataFrame(joint_results)

display(p3_table)


,Joint Event,Count,Probability
0,P(S ∩ H),122,0.018959
1,P(S ∩ Q),333,0.051748
2,P(H ∩ Q),180,0.027972
3,P(S ∩ C),185,0.028749


In [104]:

# P3 – Independent Verification (R5)

verification_results = []

for A, B in joint_events:

    # Four mutually exclusive groups
    both = int((ev[A] & ev[B]).sum())
    a_only = int((ev[A] & ~ev[B]).sum())
    b_only = int((~ev[A] & ev[B]).sum())
    neither = int((~ev[A] & ~ev[B]).sum())

    # Verify all observations are accounted for
    total = both + a_only + b_only + neither

    # Calculate probability and complement
    p_joint = both / N
    p_complement = (a_only + b_only + neither) / N

    assert total == N
    assert np.isclose(p_joint + p_complement, 1.0)

    # Additional check using the individual event counts
    assert both + a_only == int(ev[A].sum())
    assert both + b_only == int(ev[B].sum())

    verification_results.append({
        "Joint Event": f"P({A} ∩ {B})",
        "Joint Count": both,
        "Complement Count": a_only + b_only + neither,
        "Total": total,
        "Probability Sum": round(p_joint + p_complement, 6)
    })

p3_verification = pd.DataFrame(verification_results)

display(p3_verification)

print("P3 Verification Passed! All joint probabilities verified.")


,Joint Event,Joint Count,Complement Count,Total,Probability Sum
0,P(S ∩ H),122,6313,6435,1.0
1,P(S ∩ Q),333,6102,6435,1.0
2,P(H ∩ Q),180,6255,6435,1.0
3,P(S ∩ C),185,6250,6435,1.0


P3 Verification Passed! All joint probabilities verified.



### P3 – Mathematical Calculations

**1. Strong Week and Holiday Week**

There are 122 store-weeks that are both strong and holiday weeks.

$$
P(S \cap H)=\frac{122}{6435}
$$

$$
\boxed{P(S \cap H)=0.01896}
$$

**2. Strong Week and November–December**

There are 333 strong store-weeks in November or December.

$$
P(S \cap Q)=\frac{333}{6435}
$$

$$
\boxed{P(S \cap Q)=0.05175}
$$

**3. Holiday Week and November–December**

There are 180 holiday store-weeks in November or December.

$$
P(H \cap Q)=\frac{180}{6435}
$$

$$
\boxed{P(H \cap Q)=0.02797}
$$

**4. Strong Week and Cold Week**

There are 185 store-weeks that are both strong and cold.

$$
P(S \cap C)=\frac{185}{6435}
$$

$$
\boxed{P(S \cap C)=0.02875}
$$



### P3 – Verification (R5)

Each joint probability is verified using its complement.

For any two events A and B:

$$
P(A \cap B)+P((A \cap B)^c)=1
$$

**1. Strong Week and Holiday Week**

$$
\frac{122}{6435}+\frac{6313}{6435}=1
$$

**2. Strong Week and November–December**

$$
\frac{333}{6435}+\frac{6102}{6435}=1
$$

**3. Holiday Week and November–December**

$$
\frac{180}{6435}+\frac{6255}{6435}=1
$$

**4. Strong Week and Cold Week**

$$
\frac{185}{6435}+\frac{6250}{6435}=1
$$

All four joint probabilities satisfy the complement rule.

The Python verification also confirms that the four mutually exclusive combinations of each event pair account for all 6,435 observations.



### P3 – Interpretation

The joint probabilities measure how often two events occur together in the Walmart dataset.

Approximately 1.90% of store-weeks are both strong and holiday weeks, while 5.17% are both strong and in November–December.

Additionally, 2.80% of store-weeks are holiday weeks in November–December, and 2.87% are both strong and cold weeks.

These results describe the frequency of joint events across all 6,435 store-weeks. All four probabilities have been verified using the complement rule and independent counting.



## Part B – Conditional Probability and Bayes' Theorem

### B1 – Holidays: Conditional Probability

In this task, we calculate the probability of a strong sales week during holiday and non-holiday weeks.

The conditional probability formula is:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
$$

Using counts, the probabilities are:

$$
P(S\mid H)=\frac{\text{Strong holiday weeks}}{\text{Total holiday weeks}}
$$

$$
P(S\mid H^c)=\frac{\text{Strong non-holiday weeks}}{\text{Total non-holiday weeks}}
$$

where:

- S = Strong sales week
- H = Holiday week
- H^c = Non-holiday week

Both probabilities are calculated by counting within their respective groups.


In [105]:

# B1 – Conditional Probability: Holiday vs Non-Holiday

# Total holiday and non-holiday weeks
holiday_count = int(ev["H"].sum())
nonholiday_count = int((~ev["H"]).sum())

# Strong weeks in each group
strong_holiday = int((ev["S"] & ev["H"]).sum())
strong_nonholiday = int((ev["S"] & ~ev["H"]).sum())

# Conditional probabilities
p_S_given_H = strong_holiday / holiday_count
p_S_given_not_H = strong_nonholiday / nonholiday_count

print("Total holiday weeks:", holiday_count)
print("Total non-holiday weeks:", nonholiday_count)
print("Strong holiday weeks:", strong_holiday)
print("Strong non-holiday weeks:", strong_nonholiday)

print("\nConditional Probabilities:")
print(f"P(S | H) = {p_S_given_H:.6f}")
print(f"P(S | H^c) = {p_S_given_not_H:.6f}")

print("\nPercentages:")
print(f"Holiday weeks: {p_S_given_H * 100:.2f}%")
print(f"Non-holiday weeks: {p_S_given_not_H * 100:.2f}%")


Total holiday weeks: 450
Total non-holiday weeks: 5985
Strong holiday weeks: 122
Strong non-holiday weeks: 692

Conditional Probabilities:
P(S | H) = 0.271111
P(S | H^c) = 0.115622

Percentages:
Holiday weeks: 27.11%
Non-holiday weeks: 11.56%



### B1 – Mathematical Calculations

**1. Probability of a Strong Week Given a Holiday**

There are 450 holiday store-weeks, of which 122 are strong.

$$
P(S\mid H)=\frac{122}{450}
$$

$$
\boxed{P(S\mid H)=0.2711=27.11\%}
$$

**2. Probability of a Strong Week Given a Non-Holiday**

There are 5,985 non-holiday store-weeks, of which 692 are strong.

$$
P(S\mid H^c)=\frac{692}{5985}
$$

$$
\boxed{P(S\mid H^c)=0.1156=11.56\%}
$$

**Comparison**

$$
\frac{P(S\mid H)}{P(S\mid H^c)}
=\frac{0.2711}{0.1156}\approx2.34
$$

A strong sales week is approximately 2.34 times as likely during a holiday week as during a non-holiday week.


In [106]:

# B1 – Verification (R5)

# Complement probabilities within each group
p_not_S_given_H = int((~ev["S"] & ev["H"]).sum()) / holiday_count
p_not_S_given_not_H = int((~ev["S"] & ~ev["H"]).sum()) / nonholiday_count

# Complement checks
check_holiday = p_S_given_H + p_not_S_given_H
check_nonholiday = p_S_given_not_H + p_not_S_given_not_H

assert np.isclose(check_holiday, 1.0)
assert np.isclose(check_nonholiday, 1.0)

# Independent verification using total probability
p_H = holiday_count / len(ev)
p_not_H = nonholiday_count / len(ev)

p_S_rebuilt = (
    p_S_given_H * p_H
    + p_S_given_not_H * p_not_H
)

p_S_direct = ev["S"].mean()

assert np.isclose(p_S_rebuilt, p_S_direct)

print("Holiday complement check:", round(check_holiday, 6))
print("Non-holiday complement check:", round(check_nonholiday, 6))
print("Rebuilt P(S):", round(p_S_rebuilt, 6))
print("Direct P(S):", round(p_S_direct, 6))

print("\nB1 Verification Passed! All probability checks are correct.")


Holiday complement check: 1.0
Non-holiday complement check: 1.0
Rebuilt P(S): 0.126496
Direct P(S): 0.126496

B1 Verification Passed! All probability checks are correct.



### B1 – Interpretation

The probability of a strong sales week is 27.11% during holiday weeks compared with 11.56% during non-holiday weeks, making strong sales approximately 2.34 times as likely during holiday weeks.

Both conditional probabilities were verified using their complements, satisfying Rule R5.



### B2 – Bayes' Theorem

In this task, we calculate the probability that a store-week is a holiday week, given that it is a strong sales week.

Bayes' theorem is:

$$
P(H\mid S)=\frac{P(S\mid H)P(H)}{P(S)}
$$

where:

- $P(H\mid S)$: Probability of a holiday week given strong sales.
- $P(S\mid H)$: Probability of strong sales given a holiday week.
- $P(H)$: Probability of a holiday week.
- $P(S)$: Probability of a strong sales week.

The result will be verified by directly counting holiday weeks among all strong store-weeks (R5).


In [107]:

# B2 – Calculate P(H | S) Using Bayes' Theorem

# Probabilities from previous sections
p_S_given_H = ev.loc[ev["H"], "S"].mean()
p_H = ev["H"].mean()
p_S = ev["S"].mean()

# Apply Bayes' theorem
p_H_given_S_bayes = (p_S_given_H * p_H) / p_S

print("P(S | H):", round(p_S_given_H, 6))
print("P(H):", round(p_H, 6))
print("P(S):", round(p_S, 6))

print("\nBayes' Theorem Result:")
print(f"P(H | S) = {p_H_given_S_bayes:.6f}")
print(f"Percentage = {p_H_given_S_bayes * 100:.2f}%")


P(S | H): 0.271111
P(H): 0.06993
P(S): 0.126496

Bayes' Theorem Result:
P(H | S) = 0.149877
Percentage = 14.99%


In [108]:

# B2 – Verification by Counting (R5)

# Count all strong weeks
total_strong = int(ev["S"].sum())

# Count strong weeks that are also holidays
strong_holiday = int((ev["S"] & ev["H"]).sum())

# Direct conditional probability
p_H_given_S_count = strong_holiday / total_strong

# Verify both methods give the same result
assert np.isclose(
    p_H_given_S_bayes,
    p_H_given_S_count
)

# Additional complement verification
strong_nonholiday = int((ev["S"] & ~ev["H"]).sum())
p_not_H_given_S = strong_nonholiday / total_strong

assert np.isclose(
    p_H_given_S_count + p_not_H_given_S,
    1.0
)

print("Total strong weeks:", total_strong)
print("Strong holiday weeks:", strong_holiday)
print("Strong non-holiday weeks:", strong_nonholiday)

print("\nVerification Results:")
print(f"Bayes' theorem: {p_H_given_S_bayes:.6f}")
print(f"Direct counting: {p_H_given_S_count:.6f}")
print(f"Complement check: {p_H_given_S_count + p_not_H_given_S:.6f}")

print("\nB2 Verification Passed! Both methods match.")


Total strong weeks: 814
Strong holiday weeks: 122
Strong non-holiday weeks: 692

Verification Results:
Bayes' theorem: 0.149877
Direct counting: 0.149877
Complement check: 1.000000

B2 Verification Passed! Both methods match.



### B2 – Mathematical Calculations

**Method 1: Bayes' Theorem**

$$
P(H\mid S)=\frac{P(S\mid H)P(H)}{P(S)}
$$

Substituting the probabilities:

$$
P(H\mid S)=\frac{\left(\frac{122}{450}\right)
\left(\frac{450}{6435}\right)}
{\frac{814}{6435}}
$$

Simplifying:

$$
P(H\mid S)=\frac{122}{814}
$$

$$
\boxed{P(H\mid S)=0.1499=14.99\%}
$$

**Method 2: Direct Counting (R5)**

There are 814 strong store-weeks, and 122 of them are holiday weeks.

$$
P(H\mid S)=\frac{122}{814}
$$

$$
\boxed{P(H\mid S)=0.1499=14.99\%}
$$

Both methods produce the same probability.

**Complement Verification**

There are 692 strong non-holiday weeks.

$$
P(H^c\mid S)=\frac{692}{814}=0.8501
$$

$$
P(H\mid S)+P(H^c\mid S)
=\frac{122+692}{814}=1
$$

Therefore, the result satisfies Rule R5.



### B2 – Interpretation

Among all strong sales weeks, approximately 14.99% occur during holiday weeks, while 85.01% occur during non-holiday weeks.

Bayes' theorem and direct counting produce the same result, confirming the calculation.



### B3 – Law of Total Probability

In this task, we divide all 6,435 store-weeks into five mutually exclusive groups:

1. Super Bowl
2. Labor Day
3. Thanksgiving
4. Christmas
5. No holiday

For each group, we calculate:

**1. Probability of the group**

$$
P(g)=\frac{\text{Number of store-weeks in group }g}{6435}
$$

**2. Probability of a strong week given the group**

$$
P(S\mid g)=\frac{\text{Strong store-weeks in group }g}{\text{Total store-weeks in group }g}
$$

**3. Share of all strong weeks belonging to the group**

$$
P(g\mid S)=\frac{\text{Strong store-weeks in group }g}{\text{Total strong store-weeks}}
$$

**Law of Total Probability**

Since the five groups are mutually exclusive and cover the entire dataset:

$$
P(S)=\sum_g P(S\mid g)P(g)
$$

We will verify that the reconstructed probability equals the directly calculated probability of a strong week, satisfying Rule R5.


In [109]:

# B3 – Classify Store-Weeks into Five Holiday Groups

holiday_dates = {
    "2010-02-12": "Super Bowl",
    "2011-02-11": "Super Bowl",
    "2012-02-10": "Super Bowl",

    "2010-09-10": "Labor Day",
    "2011-09-09": "Labor Day",
    "2012-09-07": "Labor Day",

    "2010-11-26": "Thanksgiving",
    "2011-11-25": "Thanksgiving",

    "2010-12-31": "Christmas",
    "2011-12-30": "Christmas"
}

# Create holiday-group column
ev["Holiday_Group"] = (
    ev["Date"].dt.strftime("%Y-%m-%d")
    .map(holiday_dates)
    .fillna("No holiday")
)

# Define the required group order
holiday_groups = [
    "Super Bowl",
    "Labor Day",
    "Thanksgiving",
    "Christmas",
    "No holiday"
]

# Verify group classification matches Holiday_Flag
assert (
    (ev["Holiday_Group"] != "No holiday") == ev["H"]
).all()

# Display group counts
print(ev["Holiday_Group"].value_counts().reindex(holiday_groups))

print("\nB3 Holiday Classification Passed!")


Holiday_Group
Super Bowl       135
Labor Day        135
Thanksgiving      90
Christmas         90
No holiday      5985
Name: count, dtype: int64

B3 Holiday Classification Passed!


In [110]:

# B3 – Calculate Group Probabilities

N = len(ev)
total_strong = int(ev["S"].sum())

b3_results = []

for group in holiday_groups:

    group_data = ev[ev["Holiday_Group"] == group]

    group_count = len(group_data)
    strong_count = int(group_data["S"].sum())

    # P(g)
    p_group = group_count / N

    # P(S | g)
    p_strong_given_group = strong_count / group_count

    # P(g | S)
    p_group_given_strong = strong_count / total_strong

    # Contribution to total probability
    contribution = p_group * p_strong_given_group

    b3_results.append({
        "Holiday Group": group,
        "Total Weeks": group_count,
        "Strong Weeks": strong_count,
        "P(g)": p_group,
        "P(S|g)": p_strong_given_group,
        "P(g|S)": p_group_given_strong,
        "Contribution": contribution
    })

b3_table = pd.DataFrame(b3_results)

display(
    b3_table.style.format({
        "P(g)": "{:.6f}",
        "P(S|g)": "{:.6f}",
        "P(g|S)": "{:.6f}",
        "Contribution": "{:.6f}"
    })
)


,Holiday Group,Total Weeks,Strong Weeks,P(g),P(S|g),P(g|S),Contribution
0,Super Bowl,135,26,0.020979,0.192593,0.031941,0.004040
1,Labor Day,135,16,0.020979,0.118519,0.019656,0.002486
2,Thanksgiving,90,74,0.013986,0.822222,0.090909,0.011500
3,Christmas,90,6,0.013986,0.066667,0.007371,0.000932
4,No holiday,5985,692,0.930070,0.115622,0.850123,0.107537


In [111]:

# B3 – Law of Total Probability Verification (R5)

# Reconstruct P(S) from five groups
p_S_rebuilt = b3_table["Contribution"].sum()

# Direct probability from full dataset
p_S_direct = ev["S"].mean()

# Verify all group probabilities sum to 1
group_probability_sum = b3_table["P(g)"].sum()

# Verify the shares of strong weeks sum to 1
strong_share_sum = b3_table["P(g|S)"].sum()

# Verify counts
assert b3_table["Total Weeks"].sum() == len(ev)
assert b3_table["Strong Weeks"].sum() == total_strong

# Probability checks
assert np.isclose(group_probability_sum, 1.0)
assert np.isclose(strong_share_sum, 1.0)
assert np.isclose(p_S_rebuilt, p_S_direct)

print("Total store-weeks:", b3_table["Total Weeks"].sum())
print("Total strong weeks:", b3_table["Strong Weeks"].sum())

print("\nProbability Verification:")
print(f"Sum of P(g): {group_probability_sum:.6f}")
print(f"Sum of P(g|S): {strong_share_sum:.6f}")
print(f"Rebuilt P(S): {p_S_rebuilt:.6f}")
print(f"Direct P(S): {p_S_direct:.6f}")

# Identify the highest conditional probability
holiday_only = b3_table[
    b3_table["Holiday Group"] != "No holiday"
]

highest_holiday = holiday_only.loc[
    holiday_only["P(S|g)"].idxmax(),
    "Holiday Group"
]

largest_share_group = b3_table.loc[
    b3_table["P(g|S)"].idxmax(),
    "Holiday Group"
]

print("\nResults:")
print("Holiday with highest P(S|g):", highest_holiday)
print("Group with largest P(g|S):", largest_share_group)

print("\nB3 Verification Passed! All checks are correct.")


Total store-weeks: 6435
Total strong weeks: 814

Probability Verification:
Sum of P(g): 1.000000
Sum of P(g|S): 1.000000
Rebuilt P(S): 0.126496
Direct P(S): 0.126496

Results:
Holiday with highest P(S|g): Thanksgiving
Group with largest P(g|S): No holiday

B3 Verification Passed! All checks are correct.



### B3 – Mathematical Calculations

**1. Probability of each holiday group**

$$
P(\text{Super Bowl})=\frac{135}{6435}=0.020979
$$

$$
P(\text{Labor Day})=\frac{135}{6435}=0.020979
$$

$$
P(\text{Thanksgiving})=\frac{90}{6435}=0.013986
$$

$$
P(\text{Christmas})=\frac{90}{6435}=0.013986
$$

$$
P(\text{No holiday})=\frac{5985}{6435}=0.930070
$$

**2. Probability of a strong week within each group**

$$
P(S\mid\text{Super Bowl})=\frac{26}{135}=0.192593
$$

$$
P(S\mid\text{Labor Day})=\frac{16}{135}=0.118519
$$

$$
P(S\mid\text{Thanksgiving})=\frac{74}{90}=0.822222
$$

$$
P(S\mid\text{Christmas})=\frac{6}{90}=0.066667
$$

$$
P(S\mid\text{No holiday})=\frac{692}{5985}=0.115622
$$

**3. Rebuilding P(S) using the Law of Total Probability**

$$
P(S)=\sum_g P(S\mid g)P(g)
$$

Substituting:

$$
\begin{aligned}
P(S)={}&
\left(\frac{26}{135}\times\frac{135}{6435}\right)\\
&+\left(\frac{16}{135}\times\frac{135}{6435}\right)\\
&+\left(\frac{74}{90}\times\frac{90}{6435}\right)\\
&+\left(\frac{6}{90}\times\frac{90}{6435}\right)\\
&+\left(\frac{692}{5985}\times\frac{5985}{6435}\right)
\end{aligned}
$$

Simplifying:

$$
P(S)=\frac{26+16+74+6+692}{6435}
$$

$$
P(S)=\frac{814}{6435}
$$

$$
\boxed{P(S)=0.126496=12.65\%}
$$

**4. Verification (R5)**

From P2, the direct probability was:

$$
P(S)=\frac{814}{6435}=0.126496
$$

Therefore:

$$
P(S)_{\text{rebuilt}}=P(S)_{\text{direct}}
$$

Both methods give the same result.

Additionally:

$$
\sum_g P(g)=1
$$

$$
\sum_g P(g\mid S)=1
$$

All checks are satisfied.

**5. Highest probability of strong sales**

Thanksgiving has the highest conditional probability:

$$
P(S\mid\text{Thanksgiving})=\frac{74}{90}=82.22\%
$$

**6. Group contributing the largest share of strong weeks**

The no-holiday group contains 692 of the 814 strong store-weeks.

$$
P(\text{No holiday}\mid S)=\frac{692}{814}
$$

$$
\boxed{P(\text{No holiday}\mid S)=0.8501=85.01\%}
$$



### B3 – Interpretation

Thanksgiving has the highest probability of a strong sales week at 82.22%. However, non-holiday weeks account for the largest share of all strong weeks at 85.01%, because they make up most of the dataset.

The law of total probability reconstructs P(S) = 0.126496, matching the direct calculation from P2. All probability checks passed, satisfying Rule R5.



### B4 – Holiday Season: Conditional Probability

In this task, we calculate two conditional probabilities:

1. P(S | Q): The probability of a strong sales week given that the week is in November or December.
2. P(Q | S): The probability that a week is in November or December given that it is a strong sales week.

Using conditional probability:

$$
P(S\mid Q)=\frac{P(S\cap Q)}{P(Q)}
$$

$$
P(Q\mid S)=\frac{P(S\cap Q)}{P(S)}
$$

The two probabilities have the same joint event in the numerator but different conditioning events in the denominator.


In [112]:

# B4 – Holiday Season Conditional Probabilities

# Count all November–December weeks
season_count = int(ev["Q"].sum())

# Count all strong weeks
strong_count = int(ev["S"].sum())

# Count strong weeks in November–December
strong_season_count = int((ev["S"] & ev["Q"]).sum())

# Conditional probabilities
p_S_given_Q = strong_season_count / season_count
p_Q_given_S = strong_season_count / strong_count

print("Total November–December weeks:", season_count)
print("Total strong weeks:", strong_count)
print("Strong November–December weeks:", strong_season_count)

print("\nConditional Probabilities:")
print(f"P(S | Q) = {p_S_given_Q:.6f}")
print(f"P(Q | S) = {p_Q_given_S:.6f}")

print("\nPercentages:")
print(f"P(S | Q) = {p_S_given_Q * 100:.2f}%")
print(f"P(Q | S) = {p_Q_given_S * 100:.2f}%")


Total November–December weeks: 810
Total strong weeks: 814
Strong November–December weeks: 333

Conditional Probabilities:
P(S | Q) = 0.411111
P(Q | S) = 0.409091

Percentages:
P(S | Q) = 41.11%
P(Q | S) = 40.91%



### B4 – Mathematical Calculations

**1. Probability of a Strong Week Given November–December**

There are 810 November–December store-weeks, of which 333 are strong.

$$
P(S\mid Q)=\frac{333}{810}
$$

$$
\boxed{P(S\mid Q)=0.4111=41.11\%}
$$

**2. Probability of November–December Given a Strong Week**

There are 814 strong store-weeks, of which 333 occur in November or December.

$$
P(Q\mid S)=\frac{333}{814}
$$

$$
\boxed{P(Q\mid S)=0.4091=40.91\%}
$$

**Comparison**

Both probabilities use the same joint count of 333, but they have different denominators:

$$
P(S\mid Q)=\frac{333}{810}
$$

$$
P(Q\mid S)=\frac{333}{814}
$$

They are different because P(S | Q) considers only November–December weeks, while P(Q | S) considers only strong sales weeks.


In [113]:

# B4 – Verification (R5)

# Complement: non-strong weeks in Nov–Dec
not_strong_season = int((~ev["S"] & ev["Q"]).sum())

p_not_S_given_Q = not_strong_season / season_count

# Complement: strong weeks outside Nov–Dec
strong_not_season = int((ev["S"] & ~ev["Q"]).sum())

p_not_Q_given_S = strong_not_season / strong_count

# Check complements sum to 1
check_1 = p_S_given_Q + p_not_S_given_Q
check_2 = p_Q_given_S + p_not_Q_given_S

assert np.isclose(check_1, 1.0)
assert np.isclose(check_2, 1.0)

# Independent Bayes' theorem verification
p_Q = ev["Q"].mean()
p_S = ev["S"].mean()

p_Q_given_S_bayes = (p_S_given_Q * p_Q) / p_S

assert np.isclose(p_Q_given_S, p_Q_given_S_bayes)

print("Non-strong November–December weeks:", not_strong_season)
print("Strong weeks outside November–December:", strong_not_season)

print("\nComplement Verification:")
print(f"P(S|Q) + P(S^c|Q) = {check_1:.6f}")
print(f"P(Q|S) + P(Q^c|S) = {check_2:.6f}")

print("\nBayes' Theorem Verification:")
print(f"Direct P(Q|S): {p_Q_given_S:.6f}")
print(f"Bayes P(Q|S): {p_Q_given_S_bayes:.6f}")

print("\nB4 Verification Passed! All checks are correct.")


Non-strong November–December weeks: 477
Strong weeks outside November–December: 481

Complement Verification:
P(S|Q) + P(S^c|Q) = 1.000000
P(Q|S) + P(Q^c|S) = 1.000000

Bayes' Theorem Verification:
Direct P(Q|S): 0.409091
Bayes P(Q|S): 0.409091

B4 Verification Passed! All checks are correct.



### B4 – Verification (R5)

**1. Verify P(S | Q) using its complement**

Among the 810 November–December weeks:

- 333 are strong.
- 477 are not strong.

$$
P(S\mid Q)+P(S^c\mid Q)
=\frac{333}{810}+\frac{477}{810}
$$

$$
=\frac{810}{810}=1
$$

**2. Verify P(Q | S) using its complement**

Among the 814 strong weeks:

- 333 occur in November–December.
- 481 occur outside November–December.

$$
P(Q\mid S)+P(Q^c\mid S)
=\frac{333}{814}+\frac{481}{814}
$$

$$
=\frac{814}{814}=1
$$

**3. Verify using Bayes' theorem**

$$
P(Q\mid S)=\frac{P(S\mid Q)P(Q)}{P(S)}
$$

$$
=\frac{\left(\frac{333}{810}\right)
\left(\frac{810}{6435}\right)}
{\frac{814}{6435}}
$$

$$
=\frac{333}{814}=0.409091
$$

The counting and Bayes' theorem results match, and both complement checks equal 1.



### B4 – Interpretation

The probability of a strong sales week during November–December is 41.11%, while 40.91% of all strong weeks occur in November–December.

These probabilities differ because P(S | Q) considers only November–December weeks, whereas P(Q | S) considers only strong sales weeks. Both probabilities were verified using complements and Bayes' theorem.



### B5 – Staffing Rule Comparison: Precision and Recall

In this task, we compare two staffing rules:

1. **Holiday Rule (H):** Add extra staff only during holiday weeks.
2. **November–December Rule (Q):** Add extra staff during November and December.

We evaluate both rules using precision and recall.

**Precision**

Precision measures the proportion of weeks receiving extra staff that actually have strong sales.

For the holiday rule:

$$
\text{Precision}_H=P(S\mid H)
$$

For the November–December rule:

$$
\text{Precision}_Q=P(S\mid Q)
$$

**Recall**

Recall measures the proportion of actual strong weeks that receive extra staff.

For the holiday rule:

$$
\text{Recall}_H=P(H\mid S)
$$

For the November–December rule:

$$
\text{Recall}_Q=P(Q\mid S)
$$

**Bayes' Theorem**

The precision of each rule can be calculated using:

$$
P(S\mid R)=
\frac{P(R\mid S)P(S)}
{P(R\mid S)P(S)+P(R\mid S^c)P(S^c)}
$$

where R represents either staffing rule, H or Q.

Both rules will be evaluated using Bayes' theorem and verified by direct counting (R5).


In [114]:

# B5 – Staffing Rule Comparison Using Bayes' Theorem

# Prior probabilities
p_S = ev["S"].mean()
p_not_S = (~ev["S"]).mean()

# Store results for both staffing rules
b5_results = []

for rule, name in [
    ("H", "Holiday Rule"),
    ("Q", "November–December Rule")
]:
    # Likelihoods calculated by counting
    p_rule_given_S = ev.loc[ev["S"], rule].mean()
    p_rule_given_not_S = ev.loc[~ev["S"], rule].mean()

    # Precision using Bayes' theorem
    numerator = p_rule_given_S * p_S

    denominator = (
        p_rule_given_S * p_S
        + p_rule_given_not_S * p_not_S
    )

    precision_bayes = numerator / denominator

    # Recall
    recall = p_rule_given_S

    # Fraction of strong weeks missed
    missed_fraction = 1 - recall

    b5_results.append({
        "Rule": name,
        "P(R|S)": p_rule_given_S,
        "P(R|S^c)": p_rule_given_not_S,
        "Precision": precision_bayes,
        "Recall": recall,
        "Missed Fraction": missed_fraction
    })

b5_table = pd.DataFrame(b5_results)

display(
    b5_table.style.format({
        "P(R|S)": "{:.6f}",
        "P(R|S^c)": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "Missed Fraction": "{:.6f}"
    })
)


,Rule,P(R|S),P(R|S^c),Precision,Recall,Missed Fraction
0,Holiday Rule,0.149877,0.058353,0.271111,0.149877,0.850123
1,November–December Rule,0.409091,0.084860,0.411111,0.409091,0.590909


In [115]:

# B5 – Direct Counting Verification (R5)

verification_results = []

for rule, name in [
    ("H", "Holiday Rule"),
    ("Q", "November–December Rule")
]:
    # Confusion matrix counts
    TP = int((ev["S"] & ev[rule]).sum())
    FP = int((~ev["S"] & ev[rule]).sum())
    FN = int((ev["S"] & ~ev[rule]).sum())
    TN = int((~ev["S"] & ~ev[rule]).sum())

    # Precision and recall by counting
    precision_count = TP / (TP + FP)
    recall_count = TP / (TP + FN)

    # Results previously obtained with Bayes
    row = b5_table.loc[b5_table["Rule"] == name].iloc[0]

    # Independent comparison
    assert np.isclose(precision_count, row["Precision"])
    assert np.isclose(recall_count, row["Recall"])

    # Verify counts and complements
    assert TP + FP + FN + TN == len(ev)
    assert TP + FN == int(ev["S"].sum())
    assert np.isclose(recall_count + FN / (TP + FN), 1.0)

    verification_results.append({
        "Rule": name,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Precision": precision_count,
        "Recall": recall_count
    })

b5_verification = pd.DataFrame(verification_results)

display(
    b5_verification.style.format({
        "Precision": "{:.6f}",
        "Recall": "{:.6f}"
    })
)

print("B5 Verification Passed! Bayes and counting match.")


,Rule,TP,FP,FN,TN,Precision,Recall
0,Holiday Rule,122,328,692,5293,0.271111,0.149877
1,November–December Rule,333,477,481,5144,0.411111,0.409091


B5 Verification Passed! Bayes and counting match.



### B5 – Mathematical Calculations

#### 1. Holiday Staffing Rule (H)

**Precision using Bayes' Theorem**

The prior probabilities are:

$$
P(S)=\frac{814}{6435}, \quad
P(S^c)=\frac{5621}{6435}
$$

The likelihoods are:

$$
P(H\mid S)=\frac{122}{814}=0.149877
$$

$$
P(H\mid S^c)=\frac{328}{5621}=0.058353
$$

Applying Bayes' theorem:

$$
P(S\mid H)=
\frac{P(H\mid S)P(S)}
{P(H\mid S)P(S)+P(H\mid S^c)P(S^c)}
$$

Substituting:

$$
P(S\mid H)=
\frac{
\left(\frac{122}{814}\right)
\left(\frac{814}{6435}\right)
}{
\left(\frac{122}{814}\right)
\left(\frac{814}{6435}\right)
+
\left(\frac{328}{5621}\right)
\left(\frac{5621}{6435}\right)
}
$$

Simplifying:

$$
P(S\mid H)=\frac{122}{122+328}
$$

$$
\boxed{\text{Precision}_H=\frac{122}{450}=27.11\%}
$$

**Verification by Direct Counting (R5)**

$$
\text{Precision}_H=\frac{TP}{TP+FP}
$$

$$
=\frac{122}{122+328}=0.271111
$$

Both methods produce the same result.

**Recall**

$$
\text{Recall}_H=P(H\mid S)
$$

$$
=\frac{122}{814}
$$

$$
\boxed{\text{Recall}_H=14.99\%}
$$

**Fraction of Strong Weeks Missed**

$$
1-\text{Recall}_H
=1-\frac{122}{814}
$$

$$
=\frac{692}{814}
$$

$$
\boxed{\text{Missed}_H=85.01\%}
$$

#### 2. November–December Staffing Rule (Q)

**Precision using Bayes' Theorem**

The likelihoods are:

$$
P(Q\mid S)=\frac{333}{814}=0.409091
$$

$$
P(Q\mid S^c)=\frac{477}{5621}=0.084860
$$

Applying Bayes' theorem:

$$
P(S\mid Q)=
\frac{P(Q\mid S)P(S)}
{P(Q\mid S)P(S)+P(Q\mid S^c)P(S^c)}
$$

Substituting:

$$
P(S\mid Q)=
\frac{
\left(\frac{333}{814}\right)
\left(\frac{814}{6435}\right)
}{
\left(\frac{333}{814}\right)
\left(\frac{814}{6435}\right)
+
\left(\frac{477}{5621}\right)
\left(\frac{5621}{6435}\right)
}
$$

Simplifying:

$$
P(S\mid Q)=\frac{333}{333+477}
$$

$$
\boxed{\text{Precision}_Q=\frac{333}{810}=41.11\%}
$$

**Verification by Direct Counting (R5)**

$$
\text{Precision}_Q=\frac{TP}{TP+FP}
$$

$$
=\frac{333}{333+477}=0.411111
$$

Both methods match.

**Recall**

$$
\text{Recall}_Q=P(Q\mid S)
$$

$$
=\frac{333}{814}
$$

$$
\boxed{\text{Recall}_Q=40.91\%}
$$

**Fraction of Strong Weeks Missed**

$$
1-\text{Recall}_Q
=1-\frac{333}{814}
$$

$$
=\frac{481}{814}
$$

$$
\boxed{\text{Missed}_Q=59.09\%}
$$

#### 3. Comparison of Both Staffing Rules

| Metric | Holiday Rule | November–December Rule |
|---|---:|---:|
| Precision | 27.11% | 41.11% |
| Recall | 14.99% | 40.91% |
| Strong weeks identified | 122 | 333 |
| Strong weeks missed | 692 | 481 |
| Fraction missed | 85.01% | 59.09% |
| Weeks receiving extra staff | 450 | 810 |

The November–December rule has higher precision and recall, but it requires extra staffing during more weeks.



### B5 – Verification (R5)

Both staffing rules were verified using Bayes' theorem and direct counting.

**Holiday Rule**

$$
P(S\mid H)_{\text{Bayes}}
=P(S\mid H)_{\text{Counting}}
=\frac{122}{450}=0.271111
$$

Recall and missed fraction:

$$
\frac{122}{814}+\frac{692}{814}=1
$$

**November–December Rule**

$$
P(S\mid Q)_{\text{Bayes}}
=P(S\mid Q)_{\text{Counting}}
=\frac{333}{810}=0.411111
$$

Recall and missed fraction:

$$
\frac{333}{814}+\frac{481}{814}=1
$$

The confusion-matrix counts also sum to 6,435 for each rule.

Therefore, both staffing rules satisfy Rule R5.



### B5 – Interpretation

The holiday staffing rule has 27.11% precision and 14.99% recall, missing 85.01% of strong sales weeks.

The November–December rule performs better, with 41.11% precision and 40.91% recall, missing 59.09% of strong weeks.

Therefore, the November–December rule identifies more strong sales weeks and has higher precision, although it requires extra staffing during more weeks.



### B6 – Base Rate and Holiday Rule Precision

In this task, we examine how changing the base rate of strong sales weeks affects the precision of the holiday staffing rule.

The base rate is the overall probability of a strong sales week:

$$
P(S)=\frac{814}{6435}=0.126496
$$

We consider two hypothetical scenarios:

1. Strong weeks are twice as common.
2. Strong weeks are half as common.

We keep the conditional likelihoods from B5 unchanged:

$$
P(H\mid S)=\frac{122}{814}=0.149877
$$

$$
P(H\mid S^c)=\frac{328}{5621}=0.058353
$$

Using Bayes' theorem, the precision of the holiday rule is:

$$
P(S\mid H)=
\frac{P(H\mid S)P(S)}
{P(H\mid S)P(S)+P(H\mid S^c)P(S^c)}
$$

We will substitute the new base rates into this formula and calculate the resulting precision for both scenarios.


In [116]:

# B6 – Base Rate Analysis

# Original base rate
p_S_original = ev["S"].mean()

# Conditional likelihoods from B5
p_H_given_S = ev.loc[ev["S"], "H"].mean()
p_H_given_not_S = ev.loc[~ev["S"], "H"].mean()

# New hypothetical base rates
p_S_double = 2 * p_S_original
p_S_half = p_S_original / 2

# Function to calculate precision using Bayes' theorem
def holiday_precision(base_rate):
    numerator = p_H_given_S * base_rate

    denominator = (
        p_H_given_S * base_rate
        + p_H_given_not_S * (1 - base_rate)
    )

    return numerator / denominator

# Calculate precision for each scenario
precision_original = holiday_precision(p_S_original)
precision_double = holiday_precision(p_S_double)
precision_half = holiday_precision(p_S_half)

print("Original P(S):", round(p_S_original, 6))
print("Doubled P(S):", round(p_S_double, 6))
print("Halved P(S):", round(p_S_half, 6))

print("\nHoliday Rule Precision:")
print(f"Original: {precision_original:.6f} ({precision_original * 100:.2f}%)")
print(f"Doubled:  {precision_double:.6f} ({precision_double * 100:.2f}%)")
print(f"Halved:   {precision_half:.6f} ({precision_half * 100:.2f}%)")


Original P(S): 0.126496
Doubled P(S): 0.252991
Halved P(S): 0.063248

Holiday Rule Precision:
Original: 0.271111 (27.11%)
Doubled:  0.465204 (46.52%)
Halved:   0.147789 (14.78%)


In [117]:

# B6 – Base Rate Comparison Table

b6_results = pd.DataFrame({
    "Scenario": [
        "Original",
        "Twice as Common",
        "Half as Common"
    ],
    "P(S)": [
        p_S_original,
        p_S_double,
        p_S_half
    ],
    "P(S^c)": [
        1 - p_S_original,
        1 - p_S_double,
        1 - p_S_half
    ],
    "Holiday Rule Precision": [
        precision_original,
        precision_double,
        precision_half
    ]
})

display(
    b6_results.style.format({
        "P(S)": "{:.6f}",
        "P(S^c)": "{:.6f}",
        "Holiday Rule Precision": "{:.6f}"
    })
)


,Scenario,P(S),P(S^c),Holiday Rule Precision
0,Original,0.126496,0.873504,0.271111
1,Twice as Common,0.252991,0.747009,0.465204
2,Half as Common,0.063248,0.936752,0.147789



### B6 – Mathematical Calculations

The precision of the holiday staffing rule is calculated using Bayes' theorem:

$$
P(S\mid H)=
\frac{P(H\mid S)P(S)}
{P(H\mid S)P(S)+P(H\mid S^c)P(S^c)}
$$

The conditional likelihoods remain fixed:

$$
P(H\mid S)=\frac{122}{814}
$$

$$
P(H\mid S^c)=\frac{328}{5621}
$$

#### 1. Original Base Rate

$$
P(S)=\frac{814}{6435}=0.126496
$$

$$
P(S^c)=1-P(S)=0.873504
$$

The original precision is:

$$
\boxed{P(S\mid H)=0.271111=27.11\%}
$$

#### 2. Strong Weeks Twice as Common

The new base rate is:

$$
P_{\text{double}}(S)=2\left(\frac{814}{6435}\right)
$$

$$
P_{\text{double}}(S)=0.252991
$$

The new complement is:

$$
P_{\text{double}}(S^c)=1-0.252991=0.747009
$$

Applying Bayes' theorem:

$$
P_{\text{double}}(S\mid H)=
\frac{
\left(\frac{122}{814}\right)
\left(\frac{1628}{6435}\right)
}{
\left(\frac{122}{814}\right)
\left(\frac{1628}{6435}\right)
+
\left(\frac{328}{5621}\right)
\left(\frac{4807}{6435}\right)
}
$$

Therefore:

$$
\boxed{P_{\text{double}}(S\mid H)=0.465204=46.52\%}
$$

#### 3. Strong Weeks Half as Common

The new base rate is:

$$
P_{\text{half}}(S)=\frac{1}{2}\left(\frac{814}{6435}\right)
$$

$$
P_{\text{half}}(S)=0.063248
$$

The new complement is:

$$
P_{\text{half}}(S^c)=1-0.063248=0.936752
$$

Applying Bayes' theorem:

$$
P_{\text{half}}(S\mid H)=
\frac{
\left(\frac{122}{814}\right)
\left(\frac{407}{6435}\right)
}{
\left(\frac{122}{814}\right)
\left(\frac{407}{6435}\right)
+
\left(\frac{328}{5621}\right)
\left(\frac{6028}{6435}\right)
}
$$

Therefore:

$$
\boxed{P_{\text{half}}(S\mid H)=0.147789=14.78\%}
$$


In [118]:

# B6 – Verification (R5)

# Verify original precision using direct counting
precision_count = (
    (ev["S"] & ev["H"]).sum() / ev["H"].sum()
)

assert np.isclose(precision_original, precision_count)

# Verify hypothetical scenarios using posterior odds
def verify_base_rate(base_rate, precision):
    # Posterior probability of not being strong
    numerator_not_strong = (
        p_H_given_not_S * (1 - base_rate)
    )

    denominator = (
        p_H_given_S * base_rate
        + numerator_not_strong
    )

    p_not_S_given_H = numerator_not_strong / denominator

    # Complement check
    assert np.isclose(precision + p_not_S_given_H, 1.0)

    # Independent check using Bayes' odds form
    prior_odds = base_rate / (1 - base_rate)
    likelihood_ratio = p_H_given_S / p_H_given_not_S

    posterior_odds = prior_odds * likelihood_ratio
    precision_from_odds = posterior_odds / (1 + posterior_odds)

    assert np.isclose(precision, precision_from_odds)

    return precision + p_not_S_given_H

check_original = verify_base_rate(
    p_S_original, precision_original
)

check_double = verify_base_rate(
    p_S_double, precision_double
)

check_half = verify_base_rate(
    p_S_half, precision_half
)

print("Original precision by counting:", round(precision_count, 6))
print("Original precision by Bayes:", round(precision_original, 6))

print("\nComplement Verification:")
print("Original:", round(check_original, 6))
print("Doubled:", round(check_double, 6))
print("Halved:", round(check_half, 6))

print("\nB6 Verification Passed! All probability checks are correct.")


Original precision by counting: 0.271111
Original precision by Bayes: 0.271111

Complement Verification:
Original: 1.0
Doubled: 1.0
Halved: 1.0

B6 Verification Passed! All probability checks are correct.



### B6 – Verification (R5)

The original holiday-rule precision was verified using direct counting:

$$
P(S\mid H)=\frac{122}{450}=0.271111
$$

This matches the result obtained using Bayes' theorem.

For the hypothetical scenarios, we verify that the posterior probabilities sum to 1.

**Twice as Common**

$$
P(S\mid H)+P(S^c\mid H)
$$

$$
=0.465204+0.534796=1
$$

**Half as Common**

$$
P(S\mid H)+P(S^c\mid H)
$$

$$
=0.147789+0.852211=1
$$

The Python code also independently verifies the results using the odds form of Bayes' theorem.

All probability checks are satisfied.



### B6 – Interpretation

The holiday rule's original precision is 27.11%. If strong weeks become twice as common, precision increases to 46.52%. If strong weeks become half as common, precision decreases to 14.78%.

This shows that precision depends on the base rate of strong sales weeks, even when the conditional likelihoods remain unchanged.



### B7 – Independence Checks

In this task, we investigate whether strong sales weeks (S) are independent of the other six events:

- H: Holiday week
- Q: November–December
- C: Cold week
- F: Expensive fuel
- J: High unemployment
- G: Large store

**Independence Condition**

Two events A and B are independent if:

$$
P(A\cap B)=P(A)P(B)
$$

For this project, we check:

$$
P(S\cap E)=P(S)P(E)
$$

where E represents H, Q, C, F, J, or G.

If the two probabilities are unequal, the events are not independent in the observed dataset.

We will calculate the probabilities for all six pairs, compare the results, and verify them using direct counting and complements (R5).


In [119]:

# B7 – Independence Checks

# Events to compare with strong sales weeks (S)
features = ["H", "Q", "C", "F", "J", "G"]

N = len(ev)

# Overall probability of a strong week
p_S = ev["S"].mean()

b7_results = []

for event in features:

    # Individual probability
    p_E = ev[event].mean()

    # Joint probability
    joint_count = int((ev["S"] & ev[event]).sum())
    p_joint = joint_count / N

    # Product of individual probabilities
    p_product = p_S * p_E

    # Difference
    difference = p_joint - p_product

    # Independence comparison
    independent = np.isclose(
        p_joint, p_product, atol=1e-12, rtol=0
    )

    b7_results.append({
        "Event Pair": f"(S, {event})",
        "Joint Count": joint_count,
        "P(S∩E)": p_joint,
        "P(S)P(E)": p_product,
        "Difference": difference,
        "Independent": "Yes" if independent else "No"
    })

b7_table = pd.DataFrame(b7_results)

display(
    b7_table.style.format({
        "P(S∩E)": "{:.6f}",
        "P(S)P(E)": "{:.6f}",
        "Difference": "{:+.6f}"
    })
)


,Event Pair,Joint Count,P(S∩E),P(S)P(E),Difference,Independent
0,"(S, H)",122,0.018959,0.008846,+0.010113,No
1,"(S, Q)",333,0.051748,0.015923,+0.035826,No
2,"(S, C)",185,0.028749,0.019441,+0.009308,No
3,"(S, F)",291,0.045221,0.059542,-0.014321,No
4,"(S, J)",412,0.064025,0.059405,+0.004620,No
5,"(S, G)",367,0.057032,0.061842,-0.004810,No


In [120]:

# B7 – Independent Verification (R5)

verification_results = []

for event in features:

    # Four mutually exclusive groups
    both = int((ev["S"] & ev[event]).sum())
    s_only = int((ev["S"] & ~ev[event]).sum())
    e_only = int((~ev["S"] & ev[event]).sum())
    neither = int((~ev["S"] & ~ev[event]).sum())

    total = both + s_only + e_only + neither

    # Verify the complete sample space
    assert total == N

    # Verify marginal counts
    assert both + s_only == int(ev["S"].sum())
    assert both + e_only == int(ev[event].sum())

    # Verify the joint probability using its complement
    p_joint = both / N

    p_joint_complement = (
        s_only + e_only + neither
    ) / N

    assert np.isclose(
        p_joint + p_joint_complement, 1.0
    )

    # Independent check using conditional probability
    p_s_given_e = both / (both + e_only)
    p_e = (both + e_only) / N

    assert np.isclose(p_joint, p_s_given_e * p_e)

    verification_results.append({
        "Event Pair": f"(S, {event})",
        "Both": both,
        "S Only": s_only,
        "E Only": e_only,
        "Neither": neither,
        "Total": total,
        "Check": p_joint + p_joint_complement
    })

b7_verification = pd.DataFrame(verification_results)

display(
    b7_verification.style.format({
        "Check": "{:.6f}"
    })
)

print("B7 Verification Passed! All independence calculations verified.")


,Event Pair,Both,S Only,E Only,Neither,Total,Check
0,"(S, H)",122,692,328,5293,6435,1.000000
1,"(S, Q)",333,481,477,5144,6435,1.000000
2,"(S, C)",185,629,804,4817,6435,1.000000
3,"(S, F)",291,523,2738,2883,6435,1.000000
4,"(S, J)",412,402,2610,3011,6435,1.000000
5,"(S, G)",367,447,2779,2842,6435,1.000000


B7 Verification Passed! All independence calculations verified.



### B7 – Mathematical Calculations

Two events S and E are independent if:

$$
P(S\cap E)=P(S)P(E)
$$

From P2:

$$
P(S)=\frac{814}{6435}=0.126496
$$

**1. Strong Week and Holiday Week (S, H)**

$$
P(S\cap H)=\frac{122}{6435}=0.018959
$$

$$
P(S)P(H)=\frac{814}{6435}\times\frac{450}{6435}=0.008846
$$

Since $0.018959\ne0.008846$, S and H are not independent.

**2. Strong Week and November–December (S, Q)**

$$
P(S\cap Q)=\frac{333}{6435}=0.051748
$$

$$
P(S)P(Q)=\frac{814}{6435}\times\frac{810}{6435}=0.015923
$$

Since $0.051748\ne0.015923$, S and Q are not independent.

**3. Strong Week and Cold Week (S, C)**

$$
P(S\cap C)=\frac{185}{6435}=0.028749
$$

$$
P(S)P(C)=\frac{814}{6435}\times\frac{989}{6435}=0.019441
$$

Since $0.028749\ne0.019441$, S and C are not independent.

**4. Strong Week and Expensive Fuel (S, F)**

$$
P(S\cap F)=\frac{291}{6435}=0.045221
$$

$$
P(S)P(F)=\frac{814}{6435}\times\frac{3029}{6435}=0.059542
$$

Since $0.045221\ne0.059542$, S and F are not independent.

**5. Strong Week and High Unemployment (S, J)**

$$
P(S\cap J)=\frac{412}{6435}=0.064025
$$

$$
P(S)P(J)=\frac{814}{6435}\times\frac{3022}{6435}=0.059405
$$

Since $0.064025\ne0.059405$, S and J are not independent.

**6. Strong Week and Large Store (S, G)**

$$
P(S\cap G)=\frac{367}{6435}=0.057032
$$

$$
P(S)P(G)=\frac{814}{6435}\times\frac{3146}{6435}=0.061842
$$

Since $0.057032\ne0.061842$, S and G are not independent.

**Conclusion**

All six pairs fail the independence condition in the observed dataset. Therefore, holiday status, season, temperature, fuel price, unemployment, and store size all contain some information about strong sales weeks.



### B7 – Verification (R5)

For each pair of events S and E, the sample space is divided into four mutually exclusive groups:

$$
S\cap E,\quad S\cap E^c,\quad S^c\cap E,\quad S^c\cap E^c
$$

The probabilities of these four groups must sum to 1.

For example, for S and H:

$$
P(S\cap H)+P(S\cap H^c)+P(S^c\cap H)+P(S^c\cap H^c)=1
$$

Substituting the observed counts:

$$
\frac{122+692+328+5293}{6435}
=\frac{6435}{6435}=1
$$

We also verify the joint probability through conditional probability:

$$
P(S\cap H)=P(S\mid H)P(H)
$$

$$
=\frac{122}{450}\times\frac{450}{6435}
=\frac{122}{6435}
$$

The Python code performs these counting, complement, and conditional-probability checks for all six event pairs.

All verification checks passed, satisfying Rule R5.



### B7 – Interpretation

All six events (H, Q, C, F, J, and G) show some association with strong sales weeks because their joint probabilities differ from the products of their individual probabilities. Therefore, knowing these events provides information about strong sales, although the strength of the associations varies and they do not establish causation.



## Part N – Naive Bayes Classifier

### N1 – Time-Based Train/Test Split (R3)

In this section, we prepare the Walmart dataset for Naive Bayes classification.

According to Rule R3, the dataset must be divided chronologically:

- **Training set:** 2010–2011
- **Testing set:** 2012

We calculate each store's average weekly sales using only the training data to prevent data leakage.

Using these training averages, we recalculate the strong-week event (S) and large-store event (G) for both datasets.

The event definitions remain:

$$
S:\quad Weekly\_Sales \geq 1.10 \times StoreAverage
$$

$$
G:\quad StoreAverage \geq 1{,}000{,}000
$$

The store average is calculated using:

$$
\overline{x}_{store} =
\frac{\text{Total training sales for the store}}
{\text{Number of training weeks for the store}}
$$

Finally, we calculate the probability of a strong week separately for the training and testing sets.

$$
P(S)=\frac{\text{Number of strong store-weeks}}
{\text{Total number of store-weeks}}
$$


In [121]:

# N1 – Time-Based Train/Test Split (R3)

# Training data: 2010 and 2011
train_raw = df[df["Date"].dt.year <= 2011].copy()

# Testing data: 2012
test_raw = df[df["Date"].dt.year == 2012].copy()

print("Training set:")
print("Rows:", len(train_raw))
print("Stores:", train_raw["Store"].nunique())
print("Weeks:", train_raw["Date"].nunique())
print("Date range:",
      train_raw["Date"].min().date(),
      "to",
      train_raw["Date"].max().date())

print("\nTesting set:")
print("Rows:", len(test_raw))
print("Stores:", test_raw["Store"].nunique())
print("Weeks:", test_raw["Date"].nunique())
print("Date range:",
      test_raw["Date"].min().date(),
      "to",
      test_raw["Date"].max().date())


Training set:
Rows: 4500
Stores: 45
Weeks: 100
Date range: 2010-02-05 to 2011-12-30

Testing set:
Rows: 1935
Stores: 45
Weeks: 43
Date range: 2012-01-06 to 2012-10-26


In [122]:

# N1 – Calculate Store Averages Using Training Data Only

train_store_avg = (
    train_raw.groupby("Store")["Weekly_Sales"].mean()
)

print("Number of store averages:", len(train_store_avg))

print("\nTraining-only store averages (first 5 stores):")
display(train_store_avg.head().to_frame("Average Weekly Sales"))


Number of store averages: 45

Training-only store averages (first 5 stores):


,Average Weekly Sales
Store,
1,1.542008e+06
2,1.938857e+06
3,3.956230e+05
4,2.067728e+06
5,3.130685e+05


In [123]:

# N1 – Recalculate Events Using Training-Only Averages

# Apply the same training averages to both sets
train = add_events(train_raw, train_store_avg)
test = add_events(test_raw, train_store_avg)

print("Training data with event columns:")
display(
    train[["Store", "Date", "S", "H", "Q", "C", "F", "J", "G"]].head()
)

print("\nTesting data with event columns:")
display(
    test[["Store", "Date", "S", "H", "Q", "C", "F", "J", "G"]].head()
)


Training data with event columns:


,Store,Date,S,H,Q,C,F,J,G
0,1,2010-02-05,False,False,False,False,False,True,True
1,1,2010-02-12,False,True,False,True,False,True,True
2,1,2010-02-19,False,False,False,True,False,True,True
3,1,2010-02-26,False,False,False,False,False,True,True
4,1,2010-03-05,False,False,False,False,False,True,True



Testing data with event columns:


,Store,Date,S,H,Q,C,F,J,G
100,1,2012-01-06,False,False,False,False,False,False,True
101,1,2012-01-13,False,False,False,False,False,False,True
102,1,2012-01-20,False,False,False,False,False,False,True
103,1,2012-01-27,False,False,False,False,False,False,True
104,1,2012-02-03,False,False,False,False,False,False,True


In [124]:

# N1 – Strong Week Probabilities in Train and Test

n1_results = []

for name, data in [
    ("Training (2010–2011)", train),
    ("Testing (2012)", test)
]:
    total_weeks = len(data)
    strong_weeks = int(data["S"].sum())

    p_strong = strong_weeks / total_weeks
    p_not_strong = 1 - p_strong

    n1_results.append({
        "Dataset": name,
        "Total Weeks": total_weeks,
        "Strong Weeks": strong_weeks,
        "P(S)": p_strong,
        "P(S^c)": p_not_strong
    })

n1_table = pd.DataFrame(n1_results)

display(
    n1_table.style.format({
        "P(S)": "{:.6f}",
        "P(S^c)": "{:.6f}"
    })
)


,Dataset,Total Weeks,Strong Weeks,P(S),P(S^c)
0,Training (2010–2011),4500,574,0.127556,0.872444
1,Testing (2012),1935,255,0.131783,0.868217


In [125]:

# N1 – Verification (R3 and R5)

# R3: Verify chronological split
assert train_raw["Date"].dt.year.isin([2010, 2011]).all()
assert test_raw["Date"].dt.year.eq(2012).all()

# Check that training occurs before testing
assert train_raw["Date"].max() < test_raw["Date"].min()

# Check sizes
assert len(train) == 4500
assert len(test) == 1935
assert len(train) + len(test) == len(df)

# Verify all stores are represented
assert train["Store"].nunique() == 45
assert test["Store"].nunique() == 45

# Verify that the store averages are from training only
expected_avg = train_raw.groupby("Store")["Weekly_Sales"].mean()

assert np.allclose(
    train_store_avg.sort_index(),
    expected_avg.sort_index()
)

# Independently check S and G using training averages
for data in [train, test]:
    avg = data["Store"].map(train_store_avg)

    expected_S = data["Weekly_Sales"] >= 1.10 * avg
    expected_G = avg >= 1_000_000

    assert (data["S"] == expected_S).all()
    assert (data["G"] == expected_G).all()

# R5: Verify probabilities by direct counting and complement
for name, data in [("Training", train), ("Testing", test)]:
    strong_count = int(data["S"].sum())
    not_strong_count = int((~data["S"]).sum())

    p_strong = strong_count / len(data)
    p_not_strong = not_strong_count / len(data)

    assert strong_count + not_strong_count == len(data)
    assert np.isclose(p_strong + p_not_strong, 1.0)
    assert np.isclose(p_strong, data["S"].mean())

    print(f"{name} P(S): {p_strong:.6f}")
    print(f"{name} complement check: {p_strong + p_not_strong:.6f}")

print("\nN1 Verification Passed! R3 and R5 satisfied.")


Training P(S): 0.127556
Training complement check: 1.000000
Testing P(S): 0.131783
Testing complement check: 1.000000

N1 Verification Passed! R3 and R5 satisfied.



### N1 – Mathematical Calculations

**1. Training Set (2010–2011)**

The training dataset contains 4,500 store-weeks, of which 574 are classified as strong.

$$
P_{\text{train}}(S)=\frac{574}{4500}
$$

$$
\boxed{P_{\text{train}}(S)=0.127556=12.76\%}
$$

**Complement Verification**

$$
P_{\text{train}}(S^c)=\frac{3926}{4500}=0.872444
$$

$$
P_{\text{train}}(S)+P_{\text{train}}(S^c)
=\frac{574+3926}{4500}=1
$$

**2. Testing Set (2012)**

The testing dataset contains 1,935 store-weeks, of which 255 are classified as strong.

$$
P_{\text{test}}(S)=\frac{255}{1935}
$$

$$
\boxed{P_{\text{test}}(S)=0.131783=13.18\%}
$$

**Complement Verification**

$$
P_{\text{test}}(S^c)=\frac{1680}{1935}=0.868217
$$

$$
P_{\text{test}}(S)+P_{\text{test}}(S^c)
=\frac{255+1680}{1935}=1
$$

### N1 – Interpretation

The probability of a strong sales week is 12.76% in the training set and 13.18% in the testing set.

The two probabilities are similar, with the testing set having a slightly higher proportion of strong weeks.

Store averages were calculated using only the 2010–2011 training data and applied to both datasets. This prevents test-data leakage and satisfies Rule R3.

All probability calculations were verified using direct counting and complements, satisfying Rule R5.



### N2 – Naive Bayes: Priors and Likelihoods

In this task, we calculate the prior probabilities and feature likelihoods required to train a Naive Bayes classifier.

All calculations use only the 2010–2011 training dataset, following Rule R3.

**Class Labels**

- S = 1: Strong sales week
- S = 0: Not a strong sales week

**Six Binary Features**

- H: Holiday week
- Q: November–December
- C: Cold week
- F: Expensive fuel
- J: High unemployment
- G: Large store

Each feature can take the value 1 (True) or 0 (False).

**1. Prior Probabilities**

The prior probability of a strong week is:

$$
P(S)=\frac{\text{Number of strong training weeks}}{\text{Total training weeks}}
$$

The prior probability of a non-strong week is:

$$
P(S^c)=\frac{\text{Number of non-strong training weeks}}{\text{Total training weeks}}
$$

**2. Feature Likelihoods**

For each feature E:

$$
P(E=1\mid S)=\frac{\#(E=1,S=1)}{\#(S=1)}
$$

$$
P(E=1\mid S^c)=\frac{\#(E=1,S=0)}{\#(S=0)}
$$

The likelihoods for feature value 0 are calculated similarly.

**3. Laplace Smoothing**

For later Naive Bayes predictions, we also calculate Laplace-smoothed likelihoods with alpha = 1.

For a binary feature:

$$
P(E=v\mid S=c)=\frac{\#(E=v,S=c)+1}{\#(S=c)+2}
$$

The denominator includes 2 because each feature has two possible values: 0 and 1.

We keep the raw counting probabilities and smoothed probabilities in separate tables.


In [126]:

# N2 – Calculate Class Priors Using Training Data Only

# Total training observations
n_train = len(train)

# Count strong and non-strong weeks
n_strong = int(train["S"].sum())
n_not_strong = int((~train["S"]).sum())

# Calculate prior probabilities
prior_S = n_strong / n_train
prior_not_S = n_not_strong / n_train

# Display results
print("Total training weeks:", n_train)
print("Strong weeks:", n_strong)
print("Non-strong weeks:", n_not_strong)

print("\nPrior Probabilities:")
print(f"P(S) = {prior_S:.6f}")
print(f"P(S^c) = {prior_not_S:.6f}")

# Complement verification (R5)
assert n_strong + n_not_strong == n_train
assert np.isclose(prior_S + prior_not_S, 1.0)

print("\nPrior Verification Passed!")


Total training weeks: 4500
Strong weeks: 574
Non-strong weeks: 3926

Prior Probabilities:
P(S) = 0.127556
P(S^c) = 0.872444

Prior Verification Passed!


In [127]:

# N2 – Raw Feature Likelihoods by Counting

features = ["H", "Q", "C", "F", "J", "G"]

raw_likelihood_results = []

for feature in features:

    for class_value, class_name in [
        (True, "S"),
        (False, "S^c")
    ]:

        # Select training observations belonging to the class
        class_data = train[train["S"] == class_value]

        # Count feature values
        count_1 = int(class_data[feature].sum())
        count_0 = int((~class_data[feature]).sum())

        total_class = len(class_data)

        # Raw likelihood probabilities
        p_feature_1 = count_1 / total_class
        p_feature_0 = count_0 / total_class

        raw_likelihood_results.append({
            "Feature": feature,
            "Class": class_name,
            "Count(1)": count_1,
            "Count(0)": count_0,
            "P(E=1|Class)": p_feature_1,
            "P(E=0|Class)": p_feature_0
        })

raw_likelihood_table = pd.DataFrame(raw_likelihood_results)

display(
    raw_likelihood_table.style.format({
        "P(E=1|Class)": "{:.6f}",
        "P(E=0|Class)": "{:.6f}"
    })
)


,Feature,Class,Count(1),Count(0),P(E=1|Class),P(E=0|Class)
0,H,S,99,475,0.172474,0.827526
1,H,S^c,261,3665,0.066480,0.933520
2,Q,S,335,239,0.583624,0.416376
3,Q,S^c,475,3451,0.120988,0.879012
4,C,S,165,409,0.287456,0.712544
5,C,S^c,599,3327,0.152573,0.847427
6,F,S,108,466,0.188153,0.811847
7,F,S^c,1358,2568,0.345899,0.654101
8,J,S,317,257,0.552265,0.447735
9,J,S^c,2085,1841,0.531075,0.468925


In [128]:

# N2 – Laplace-Smoothed Feature Likelihoods

alpha = 1
num_categories = 2  # Binary features: 0 and 1

smoothed_results = []

for feature in features:

    for class_value, class_name in [
        (True, "S"),
        (False, "S^c")
    ]:

        class_data = train[train["S"] == class_value]

        count_1 = int(class_data[feature].sum())
        count_0 = int((~class_data[feature]).sum())

        total_class = len(class_data)

        # Laplace smoothing
        denominator = total_class + alpha * num_categories

        p_1_smoothed = (count_1 + alpha) / denominator
        p_0_smoothed = (count_0 + alpha) / denominator

        smoothed_results.append({
            "Feature": feature,
            "Class": class_name,
            "P(E=1|Class)": p_1_smoothed,
            "P(E=0|Class)": p_0_smoothed
        })

smoothed_likelihood_table = pd.DataFrame(smoothed_results)

display(
    smoothed_likelihood_table.style.format({
        "P(E=1|Class)": "{:.6f}",
        "P(E=0|Class)": "{:.6f}"
    })
)


,Feature,Class,P(E=1|Class),P(E=0|Class)
0,H,S,0.173611,0.826389
1,H,S^c,0.066701,0.933299
2,Q,S,0.583333,0.416667
3,Q,S^c,0.121181,0.878819
4,C,S,0.288194,0.711806
5,C,S^c,0.152749,0.847251
6,F,S,0.189236,0.810764
7,F,S^c,0.345978,0.654022
8,J,S,0.552083,0.447917
9,J,S^c,0.531059,0.468941



### N2 – Mathematical Calculations

**1. Class Prior Probabilities**

The training dataset has 4,500 observations, including 574 strong weeks and 3,926 non-strong weeks.

$$
P(S)=\frac{574}{4500}=0.127556
$$

$$
P(S^c)=\frac{3926}{4500}=0.872444
$$

**Verification**

$$
P(S)+P(S^c)=\frac{574+3926}{4500}=1
$$

**2. Raw Likelihood Example: Holiday Given Strong Sales**

Among 574 strong training weeks, 99 are holiday weeks.

$$
P(H=1\mid S)=\frac{99}{574}
$$

$$
\boxed{P(H=1\mid S)=0.172474}
$$

The complement is:

$$
P(H=0\mid S)=\frac{475}{574}=0.827526
$$

**Verification**

$$
\frac{99}{574}+\frac{475}{574}=1
$$

**3. Raw Likelihood Example: Holiday Given Non-Strong Sales**

Among 3,926 non-strong training weeks, 261 are holiday weeks.

$$
P(H=1\mid S^c)=\frac{261}{3926}
$$

$$
\boxed{P(H=1\mid S^c)=0.066480}
$$

The complement is:

$$
P(H=0\mid S^c)=\frac{3665}{3926}=0.933520
$$

**Verification**

$$
\frac{261}{3926}+\frac{3665}{3926}=1
$$

**4. Laplace Smoothing**

For binary features, we use alpha = 1:

$$
P(E=v\mid S=c)=\frac{\#(E=v,S=c)+1}{\#(S=c)+2}
$$

For holiday weeks given strong sales:

$$
P_{\text{smooth}}(H=1\mid S)
=\frac{99+1}{574+2}
$$

$$
=\frac{100}{576}
$$

$$
\boxed{P_{\text{smooth}}(H=1\mid S)=0.173611}
$$

For non-strong sales:

$$
P_{\text{smooth}}(H=1\mid S^c)
=\frac{261+1}{3926+2}
$$

$$
=\frac{262}{3928}
$$

$$
\boxed{P_{\text{smooth}}(H=1\mid S^c)=0.066701}
$$

The same formulas are applied to the remaining five features: Q, C, F, J, and G.

The complete raw and smoothed likelihood tables are displayed in the Python outputs above.


In [129]:

# N2 – Verification (R3 and R5)

# R3: Confirm training data only
assert len(train) == 4500
assert train["Date"].dt.year.isin([2010, 2011]).all()

# Confirm class counts
assert n_strong == 574
assert n_not_strong == 3926
assert n_strong + n_not_strong == len(train)

# R5: Verify priors sum to 1
assert np.isclose(prior_S + prior_not_S, 1.0)

# Verify all raw and smoothed likelihoods
for feature in features:

    for class_value, class_name in [
        (True, "S"),
        (False, "S^c")
    ]:

        class_data = train.loc[train["S"] == class_value]

        n = len(class_data)
        count_1 = int(class_data[feature].sum())
        count_0 = int((~class_data[feature]).sum())

        # Raw likelihoods from stored table
        raw_row = raw_likelihood_table.loc[
            (raw_likelihood_table["Feature"] == feature) &
            (raw_likelihood_table["Class"] == class_name)
        ].iloc[0]

        # Smoothed likelihoods from stored table
        smooth_row = smoothed_likelihood_table.loc[
            (smoothed_likelihood_table["Feature"] == feature) &
            (smoothed_likelihood_table["Class"] == class_name)
        ].iloc[0]

        # Count verification
        assert count_0 + count_1 == n
        assert raw_row["Count(1)"] == count_1
        assert raw_row["Count(0)"] == count_0

        # Raw probability checks
        assert np.isclose(
            raw_row["P(E=1|Class)"],
            count_1 / n
        )

        assert np.isclose(
            raw_row["P(E=1|Class)"] +
            raw_row["P(E=0|Class)"],
            1.0
        )

        # Smoothed probability checks
        assert np.isclose(
            smooth_row["P(E=1|Class)"],
            (count_1 + 1) / (n + 2)
        )

        assert np.isclose(
            smooth_row["P(E=1|Class)"] +
            smooth_row["P(E=0|Class)"],
            1.0
        )

print("Training data verification: Passed")
print("Class prior verification: Passed")
print("Raw likelihood verification: Passed")
print("Laplace smoothing verification: Passed")

print("\nN2 Verification Passed! R3 and R5 satisfied.")


Training data verification: Passed
Class prior verification: Passed
Raw likelihood verification: Passed
Laplace smoothing verification: Passed

N2 Verification Passed! R3 and R5 satisfied.



### N2 – Verification (R3 and R5)

**Rule R3: Training Data Only**

All prior probabilities and feature likelihoods are calculated using the 4,500 training observations from 2010–2011.

The 2012 testing observations are not used to estimate the probabilities.

**Rule R5: Probability Verification**

The class priors satisfy:

$$
P(S)+P(S^c)=1
$$

For each binary feature E and class c:

$$
P(E=1\mid S=c)+P(E=0\mid S=c)=1
$$

For example, the smoothed holiday likelihoods among strong weeks satisfy:

$$
\frac{100}{576}+\frac{476}{576}=1
$$

The Python verification checks all six features for both classes, including raw and smoothed probabilities.

All checks passed.



### N2 – Interpretation

The training data contains 12.76% strong weeks and 87.24% non-strong weeks.

The likelihood tables show how frequently each feature occurs within strong and non-strong weeks. For example, holiday weeks account for 17.25% of strong training weeks, compared with 6.65% of non-strong training weeks.

We also calculated Laplace-smoothed likelihoods using alpha = 1 to prepare for Naive Bayes prediction and comparison with scikit-learn.

All probabilities were calculated from the training data only and verified using direct counting and complements.



### N3 – Manual Naive Bayes Predictions

In this task, we use Naive Bayes to predict the probability of a strong sales week for two given profiles.

The classifier uses six binary features:

- H: Holiday week
- Q: November–December
- C: Cold week
- F: Expensive fuel
- J: High unemployment
- G: Large store

**Profile A**

Holiday week, November–December, cold, cheap fuel, low unemployment, and large store.

**Profile B**

Non-holiday, not November–December, not cold, expensive fuel, high unemployment, and small store.

**Naive Bayes Formula**

For a feature vector x, the unnormalized score for class c is:

$$
\text{Score}(c)=P(c)\prod_{i=1}^{6}P(x_i\mid c)
$$

We calculate two scores:

$$
\text{Score}(S)=P(S)\prod_{i=1}^{6}P(x_i\mid S)
$$

$$
\text{Score}(S^c)=P(S^c)\prod_{i=1}^{6}P(x_i\mid S^c)
$$

The normalized posterior probability is:

$$
P(S\mid x)=
\frac{\text{Score}(S)}
{\text{Score}(S)+\text{Score}(S^c)}
$$

We use the training priors and Laplace-smoothed feature likelihoods with alpha = 1 from N2.

The calculation assumes that the six features are conditionally independent given the class.


In [130]:

# N3 – Define the Two Required Profiles

features = ["H", "Q", "C", "F", "J", "G"]

profile_a = {
    "H": 1,
    "Q": 1,
    "C": 1,
    "F": 0,
    "J": 0,
    "G": 1
}

profile_b = {
    "H": 0,
    "Q": 0,
    "C": 0,
    "F": 1,
    "J": 1,
    "G": 0
}

profiles = {
    "Profile A": profile_a,
    "Profile B": profile_b
}

profile_table = pd.DataFrame(profiles).T
profile_table = profile_table[features]

display(profile_table)


,H,Q,C,F,J,G
Profile A,1,1,1,0,0,1
Profile B,0,0,0,1,1,0


In [131]:

# N3 – Manual Naive Bayes Score Calculation

def get_smoothed_likelihood(feature, value, class_name):

    row = smoothed_likelihood_table.loc[
        (smoothed_likelihood_table["Feature"] == feature) &
        (smoothed_likelihood_table["Class"] == class_name)
    ].iloc[0]

    if value == 1:
        return float(row["P(E=1|Class)"])
    else:
        return float(row["P(E=0|Class)"])


def calculate_nb_scores(profile):

    # Training priors from N2
    score_S = prior_S
    score_not_S = prior_not_S

    details = []

    for feature in features:

        value = profile[feature]

        # Smoothed likelihood for strong class
        likelihood_S = get_smoothed_likelihood(
            feature, value, "S"
        )

        # Smoothed likelihood for non-strong class
        likelihood_not_S = get_smoothed_likelihood(
            feature, value, "S^c"
        )

        # Multiply into the two scores
        score_S *= likelihood_S
        score_not_S *= likelihood_not_S

        details.append({
            "Feature": feature,
            "Value": value,
            "P(value|S)": likelihood_S,
            "P(value|S^c)": likelihood_not_S
        })

    # Normalize the scores
    total_score = score_S + score_not_S

    posterior_S = score_S / total_score
    posterior_not_S = score_not_S / total_score

    return {
        "Score(S)": score_S,
        "Score(S^c)": score_not_S,
        "P(S|x)": posterior_S,
        "P(S^c|x)": posterior_not_S,
        "Details": pd.DataFrame(details)
    }


# Calculate both profiles
n3_results = {}

for name, profile in profiles.items():

    result = calculate_nb_scores(profile)
    n3_results[name] = result

    print(f"\n{name}")

    display(
        result["Details"].style.format({
            "P(value|S)": "{:.6f}",
            "P(value|S^c)": "{:.6f}"
        })
    )

    print(f"Score(S): {result['Score(S)']:.10f}")
    print(f"Score(S^c): {result['Score(S^c)']:.10f}")

    print(f"P(S|x): {result['P(S|x)']:.6f}")
    print(f"P(S^c|x): {result['P(S^c|x)']:.6f}")

    print(f"Strong-week probability: {result['P(S|x)'] * 100:.2f}%")



Profile A


,Feature,Value,P(value|S),P(value|S^c)
0,H,1,0.173611,0.066701
1,Q,1,0.583333,0.121181
2,C,1,0.288194,0.152749
3,F,0,0.810764,0.654022
4,J,0,0.447917,0.468941
5,G,1,0.463542,0.492617


Score(S): 0.0006267000
Score(S^c): 0.0001627433
P(S|x): 0.793851
P(S^c|x): 0.206149
Strong-week probability: 79.39%

Profile B


,Feature,Value,P(value|S),P(value|S^c)
0,H,0,0.826389,0.933299
1,Q,0,0.416667,0.878819
2,C,0,0.711806,0.847251
3,F,1,0.189236,0.345978
4,J,1,0.552083,0.531059
5,G,0,0.536458,0.507383


Score(S): 0.0017521796
Score(S^c): 0.0565192684
P(S|x): 0.030069
P(S^c|x): 0.969931
Strong-week probability: 3.01%



### N3 – Mathematical Calculations

The training class priors from N2 are:

$$
P(S)=\frac{574}{4500}=0.127556
$$

$$
P(S^c)=\frac{3926}{4500}=0.872444
$$

We use Laplace-smoothed likelihoods with alpha = 1.

---

#### Profile A: Holiday, Nov–Dec, Cold, Cheap Fuel, Low Unemployment, Large Store

The binary feature vector is:

$$
x_A=(1,1,1,0,0,1)
$$

**1. Strong-week score**

$$
\begin{aligned}
\text{Score}_A(S)={}&P(S)
P(H=1\mid S)
P(Q=1\mid S)\\
&\times P(C=1\mid S)
P(F=0\mid S)\\
&\times P(J=0\mid S)
P(G=1\mid S)
\end{aligned}
$$

Substituting the training prior and smoothed likelihoods:

$$
\begin{aligned}
\text{Score}_A(S)={}&
\frac{574}{4500}
\times\frac{100}{576}
\times\frac{336}{576}\\
&\times\frac{166}{576}
\times\frac{467}{576}
\times\frac{258}{576}
\times\frac{267}{576}
\end{aligned}
$$

$$
\boxed{\text{Score}_A(S)=0.0006267000}
$$

**2. Non-strong-week score**

$$
\begin{aligned}
\text{Score}_A(S^c)={}&
\frac{3926}{4500}
\times\frac{262}{3928}
\times\frac{476}{3928}\\
&\times\frac{600}{3928}
\times\frac{2569}{3928}
\times\frac{1842}{3928}
\times\frac{1935}{3928}
\end{aligned}
$$

$$
\boxed{\text{Score}_A(S^c)=0.0001627433}
$$

**3. Normalize the scores**

$$
P(S\mid x_A)=
\frac{\text{Score}_A(S)}
{\text{Score}_A(S)+\text{Score}_A(S^c)}
$$

$$
=\frac{0.0006267000}
{0.0006267000+0.0001627433}
$$

$$
\boxed{P(S\mid x_A)=0.793851=79.39\%}
$$

The complementary posterior is:

$$
P(S^c\mid x_A)=0.206149=20.61\%
$$

**Interpretation:** Profile A has an estimated 79.39% probability of being a strong sales week under the fitted Naive Bayes model.

---

#### Profile B: Non-Holiday, Not Nov–Dec, Not Cold, Expensive Fuel, High Unemployment, Small Store

The binary feature vector is:

$$
x_B=(0,0,0,1,1,0)
$$

**1. Strong-week score**

$$
\begin{aligned}
\text{Score}_B(S)={}&
\frac{574}{4500}
\times\frac{476}{576}
\times\frac{240}{576}\\
&\times\frac{410}{576}
\times\frac{109}{576}
\times\frac{318}{576}
\times\frac{309}{576}
\end{aligned}
$$

$$
\boxed{\text{Score}_B(S)=0.0017521796}
$$

**2. Non-strong-week score**

$$
\begin{aligned}
\text{Score}_B(S^c)={}&
\frac{3926}{4500}
\times\frac{3666}{3928}
\times\frac{3452}{3928}\\
&\times\frac{3328}{3928}
\times\frac{1359}{3928}
\times\frac{2086}{3928}
\times\frac{1993}{3928}
\end{aligned}
$$

$$
\boxed{\text{Score}_B(S^c)=0.0565192684}
$$

**3. Normalize the scores**

$$
P(S\mid x_B)=
\frac{\text{Score}_B(S)}
{\text{Score}_B(S)+\text{Score}_B(S^c)}
$$

$$
=\frac{0.0017521796}
{0.0017521796+0.0565192684}
$$

$$
\boxed{P(S\mid x_B)=0.030069=3.01\%}
$$

The complementary posterior is:

$$
P(S^c\mid x_B)=0.969931=96.99\%
$$

**Interpretation:** Profile B has an estimated 3.01% probability of being a strong sales week under the fitted Naive Bayes model.

---

#### Comparison

| Profile | Score(S) | Score(S^c) | P(S given x) |
|---|---:|---:|---:|
| A | 0.0006267000 | 0.0001627433 | 79.39% |
| B | 0.0017521796 | 0.0565192684 | 3.01% |

Profile A has a much higher estimated probability of strong sales than Profile B.


In [132]:

# N3 – Independent Verification (R3 and R5)

# Verify we are using training data only
assert len(train) == 4500
assert train["Date"].dt.year.isin([2010, 2011]).all()

# Verify the expected feature vectors
assert list(profile_a.values()) == [1, 1, 1, 0, 0, 1]
assert list(profile_b.values()) == [0, 0, 0, 1, 1, 0]

# Verify priors directly from training counts
assert np.isclose(prior_S, train["S"].mean())
assert np.isclose(prior_not_S, (~train["S"]).mean())

# Independent score calculation using training counts
def independent_nb_calculation(profile):

    scores = {}

    for class_value, class_name in [
        (True, "S"),
        (False, "S^c")
    ]:
        class_data = train.loc[train["S"] == class_value]

        # Class prior
        score = len(class_data) / len(train)

        # Calculate smoothed likelihoods directly from counts
        for feature in features:
            value = profile[feature]

            count = int(
                (class_data[feature].astype(int) == value).sum()
            )

            likelihood = (count + 1) / (len(class_data) + 2)
            score *= likelihood

        scores[class_name] = score

    total = scores["S"] + scores["S^c"]

    return (
        scores["S"],
        scores["S^c"],
        scores["S"] / total,
        scores["S^c"] / total
    )


for name, profile in profiles.items():

    score_S, score_not_S, posterior_S, posterior_not_S = (
        independent_nb_calculation(profile)
    )

    result = n3_results[name]

    # Verify independent scores match Step 3
    assert np.isclose(score_S, result["Score(S)"])
    assert np.isclose(score_not_S, result["Score(S^c)"])

    # Verify normalized posteriors
    assert np.isclose(posterior_S, result["P(S|x)"])
    assert np.isclose(posterior_not_S, result["P(S^c|x)"])

    # R5: Complement check
    assert np.isclose(posterior_S + posterior_not_S, 1.0)

    print(f"{name}:")
    print(f"  Strong probability: {posterior_S:.6f}")
    print(f"  Non-strong probability: {posterior_not_S:.6f}")
    print(f"  Complement check: {posterior_S + posterior_not_S:.6f}")
    print("  Independent calculation: Passed\n")

print("N3 Verification Passed! R3 and R5 satisfied.")


Profile A:
  Strong probability: 0.793851
  Non-strong probability: 0.206149
  Complement check: 1.000000
  Independent calculation: Passed

Profile B:
  Strong probability: 0.030069
  Non-strong probability: 0.969931
  Complement check: 1.000000
  Independent calculation: Passed

N3 Verification Passed! R3 and R5 satisfied.



### N3 – Verification (R3 and R5)

**Rule R3: Training Data Only**

All class priors and feature likelihoods were calculated using the 2010–2011 training dataset.

The 2012 testing dataset was not used to estimate any model probabilities.

**Rule R5: Probability Verification**

The normalized probabilities for each profile must sum to 1.

**Profile A**

$$
P(S\mid x_A)+P(S^c\mid x_A)
$$

$$
=0.793851+0.206149=1
$$

**Profile B**

$$
P(S\mid x_B)+P(S^c\mid x_B)
$$

$$
=0.030069+0.969931=1
$$

The Python code also independently recalculates both class scores using the original training counts and Laplace smoothing.

Both methods produce matching results.

Therefore, the N3 calculations satisfy Rules R3 and R5.



### N3 – Interpretation

The Naive Bayes model estimates a 79.39% probability of strong sales for Profile A and a 3.01% probability for Profile B.

Profile A is therefore predicted to be a strong sales week, while Profile B is predicted to be non-strong using a 50% classification threshold.

These estimates are based on training-only priors and Laplace-smoothed likelihoods. Both predictions were verified through independent calculations and posterior normalization.



### N4 – Check the Naive Bayes Independence Assumption

Naive Bayes assumes that features are conditionally independent given the class label.

For holiday status (H) and the November–December season (Q), conditional independence given strong sales (S) would require:

$$
P(H\cap Q\mid S)=P(H\mid S)P(Q\mid S)
$$

where:

- H = Holiday week
- Q = November or December
- S = Strong sales week

We calculate both sides using only the 2010–2011 training dataset.

If the probabilities differ, H and Q are not conditionally independent given S in the observed data.

We will verify the results using direct counting and the conditional complement rule, following R3 and R5.


In [133]:

# N4 – Check Conditional Independence of H and Q Given S

# Select only strong weeks from the training dataset
strong_train = train.loc[train["S"]].copy()

# Count strong training weeks
n_strong_train = len(strong_train)

# Count holiday weeks among strong weeks
n_H_given_S = int(strong_train["H"].sum())

# Count November–December weeks among strong weeks
n_Q_given_S = int(strong_train["Q"].sum())

# Count weeks satisfying both H and Q among strong weeks
n_HQ_given_S = int(
    (strong_train["H"] & strong_train["Q"]).sum()
)

# Calculate conditional probabilities
p_H_given_S_n4 = n_H_given_S / n_strong_train
p_Q_given_S_n4 = n_Q_given_S / n_strong_train
p_HQ_given_S = n_HQ_given_S / n_strong_train

# Calculate product assuming conditional independence
p_independent_product = p_H_given_S_n4 * p_Q_given_S_n4

# Compare
difference_n4 = p_HQ_given_S - p_independent_product

print("Total strong training weeks:", n_strong_train)
print("Strong holiday weeks:", n_H_given_S)
print("Strong November–December weeks:", n_Q_given_S)
print("Strong weeks in both H and Q:", n_HQ_given_S)

print("\nConditional Probabilities:")
print(f"P(H | S) = {p_H_given_S_n4:.6f}")
print(f"P(Q | S) = {p_Q_given_S_n4:.6f}")

print("\nIndependence Comparison:")
print(f"P(H ∩ Q | S) = {p_HQ_given_S:.6f}")
print(f"P(H | S)P(Q | S) = {p_independent_product:.6f}")
print(f"Difference = {difference_n4:.6f}")


Total strong training weeks: 574
Strong holiday weeks: 99
Strong November–December weeks: 335
Strong weeks in both H and Q: 80

Conditional Probabilities:
P(H | S) = 0.172474
P(Q | S) = 0.583624

Independence Comparison:
P(H ∩ Q | S) = 0.139373
P(H | S)P(Q | S) = 0.100660
Difference = 0.038713


In [134]:

# N4 – Verification (R3 and R5)

# R3: Training data only
assert len(train) == 4500
assert train["Date"].dt.year.isin([2010, 2011]).all()

# Count all four combinations among strong weeks
both = int((strong_train["H"] & strong_train["Q"]).sum())

h_only = int((strong_train["H"] & ~strong_train["Q"]).sum())

q_only = int((~strong_train["H"] & strong_train["Q"]).sum())

neither = int((~strong_train["H"] & ~strong_train["Q"]).sum())

# Verify group counts
assert both + h_only + q_only + neither == n_strong_train

# Verify marginal counts
assert both + h_only == n_H_given_S
assert both + q_only == n_Q_given_S

# Conditional complement verification
p_complement_HQ_given_S = (
    h_only + q_only + neither
) / n_strong_train

assert np.isclose(
    p_HQ_given_S + p_complement_HQ_given_S,
    1.0
)

# Direct conditional probability verification
assert np.isclose(
    p_HQ_given_S,
    (strong_train["H"] & strong_train["Q"]).mean()
)

# Compare the two sides of independence condition
assert not np.isclose(
    p_HQ_given_S,
    p_independent_product
)

print("Both H and Q:", both)
print("H only:", h_only)
print("Q only:", q_only)
print("Neither:", neither)

print("\nVerification:")
print("Total:", both + h_only + q_only + neither)

print(
    "Conditional complement check:",
    round(p_HQ_given_S + p_complement_HQ_given_S, 6)
)

print("Conditional independence:", "Not satisfied")

print("\nN4 Verification Passed! R3 and R5 satisfied.")


Both H and Q: 80
H only: 19
Q only: 255
Neither: 220

Verification:
Total: 574
Conditional complement check: 1.0
Conditional independence: Not satisfied

N4 Verification Passed! R3 and R5 satisfied.



### N4 – Mathematical Calculations

**1. Calculate P(H ∩ Q | S)**

There are 574 strong training weeks, of which 80 are both holiday weeks and in November–December.

$$
P(H\cap Q\mid S)=\frac{80}{574}
$$

$$
\boxed{P(H\cap Q\mid S)=0.139373}
$$

**2. Calculate P(H | S)**

There are 99 holiday weeks among 574 strong training weeks.

$$
P(H\mid S)=\frac{99}{574}
$$

$$
\boxed{P(H\mid S)=0.172474}
$$

**3. Calculate P(Q | S)**

There are 335 November–December weeks among 574 strong training weeks.

$$
P(Q\mid S)=\frac{335}{574}
$$

$$
\boxed{P(Q\mid S)=0.583624}
$$

**4. Check the Independence Condition**

Conditional independence requires:

$$
P(H\cap Q\mid S)=P(H\mid S)P(Q\mid S)
$$

Calculate the right-hand side:

$$
P(H\mid S)P(Q\mid S)
=\frac{99}{574}\times\frac{335}{574}
$$

$$
\boxed{P(H\mid S)P(Q\mid S)=0.100660}
$$

Compare both sides:

$$
0.139373\neq0.100660
$$

Therefore:

$$
\boxed{H\text{ and }Q\text{ are not conditionally independent given }S}
$$

**5. Verification (R5)**

The four mutually exclusive combinations among strong training weeks are:

- Both H and Q: 80
- H only: 19
- Q only: 255
- Neither: 220

Their probabilities sum to:

$$
\frac{80+19+255+220}{574}
=\frac{574}{574}=1
$$

Also, using the complement:

$$
P(H\cap Q\mid S)+P((H\cap Q)^c\mid S)
$$

$$
=\frac{80}{574}+\frac{494}{574}=1
$$

All probability checks are satisfied.



### N4 – Interpretation

Holiday status (H) and November–December (Q) are not conditionally independent given strong sales because P(H ∩ Q | S) = 0.139373 differs from P(H | S)P(Q | S) = 0.100660.

This happens because some holidays, particularly Thanksgiving and Christmas, occur in November or December, creating a relationship between holiday status and season.

Therefore, the Naive Bayes conditional independence assumption is not fully satisfied for these features, even when considering only strong sales weeks.



### N5 – scikit-learn CategoricalNB

In this task, we train a Naive Bayes classifier using scikit-learn and compare its predicted probabilities with the manual calculations from N3.

**Model:** Categorical Naive Bayes

**Training Data:** 2010–2011 (4,500 store-weeks)

**Target Variable:** S (Strong sales week)

**Features:** H, Q, C, F, J, G

All six features are binary, taking values of 0 or 1.

We use:

- CategoricalNB with alpha = 1 for Laplace smoothing.
- Empirical class priors calculated from the training data.
- Training-only store averages for the events S and G.

The two profiles are:

**Profile A:** (1, 1, 1, 0, 0, 1)

**Profile B:** (0, 0, 0, 1, 1, 0)

The model probabilities will be compared with the manually calculated N3 posterior probabilities.

**Verification:** Both methods must produce matching probabilities, and the predicted class probabilities must sum to 1.


In [135]:

# N5 – Prepare Training Features and Target

from sklearn.naive_bayes import CategoricalNB

# Six binary input features
features = ["H", "Q", "C", "F", "J", "G"]

# Training features
X_train = train[features].astype(int)

# Target: Strong week (1) or Non-strong week (0)
y_train = train["S"].astype(int)

print("Training feature shape:", X_train.shape)
print("Training target shape:", y_train.shape)

print("\nFeature columns:")
print(X_train.columns.tolist())

print("\nTarget distribution:")
print(y_train.value_counts().sort_index())

display(X_train.head())


Training feature shape: (4500, 6)
Training target shape: (4500,)

Feature columns:
['H', 'Q', 'C', 'F', 'J', 'G']

Target distribution:
S
0    3926
1     574
Name: count, dtype: int64


,H,Q,C,F,J,G
0,0,0,0,0,1,1
1,1,0,1,0,1,1
2,0,0,1,0,1,1
3,0,0,0,0,1,1
4,0,0,0,0,1,1


In [136]:

# N5 – Train Categorical Naive Bayes Classifier

# Initialize model with Laplace smoothing
nb = CategoricalNB(
    alpha=1.0,
    fit_prior=True,
    min_categories=2
)

# Train only on 2010–2011 data
nb.fit(X_train, y_train)

print("CategoricalNB model trained successfully!")

print("\nClass labels:")
print(nb.classes_)

print("\nModel class prior probabilities:")
for class_label, log_prior in zip(
    nb.classes_,
    nb.class_log_prior_
):
    print(
        f"Class {class_label}: {np.exp(log_prior):.6f}"
    )


CategoricalNB model trained successfully!

Class labels:
[0 1]

Model class prior probabilities:
Class 0: 0.872444
Class 1: 0.127556


In [137]:

# N5 – Predict Probabilities for Profiles A and B

# Create both profiles in the same feature order
X_profiles = pd.DataFrame(
    [
        [1, 1, 1, 0, 0, 1],
        [0, 0, 0, 1, 1, 0]
    ],
    columns=features,
    index=["Profile A", "Profile B"]
)

# Predict class probabilities
probabilities = nb.predict_proba(X_profiles)

# Obtain predictions
predictions = nb.predict(X_profiles)

# Create results table
n5_results = pd.DataFrame(
    probabilities,
    columns=[
        f"P(Class={c}|x)" for c in nb.classes_
    ],
    index=X_profiles.index
)

n5_results["Predicted Class"] = predictions

display(n5_results.round(6))


,P(Class=0|x),P(Class=1|x),Predicted Class
Profile A,0.206149,0.793851,1
Profile B,0.969931,0.030069,0


In [138]:

# N5 – Compare scikit-learn with Manual N3 Calculations

comparison_results = []

# Find class positions automatically
strong_idx = list(nb.classes_).index(1)
not_strong_idx = list(nb.classes_).index(0)

for i, name in enumerate(["Profile A", "Profile B"]):

    # Manual probability from N3
    manual_probability = n3_results[name]["P(S|x)"]

    # scikit-learn probability
    sklearn_probability = probabilities[i, strong_idx]

    # Absolute difference
    difference = abs(
        manual_probability - sklearn_probability
    )

    # Verify matching results
    assert np.isclose(
        manual_probability,
        sklearn_probability,
        atol=1e-10,
        rtol=0
    )

    comparison_results.append({
        "Profile": name,
        "Manual N3": manual_probability,
        "scikit-learn N5": sklearn_probability,
        "Absolute Difference": difference,
        "Match": "Yes"
    })

n5_comparison = pd.DataFrame(comparison_results)

display(
    n5_comparison.style.format({
        "Manual N3": "{:.6f}",
        "scikit-learn N5": "{:.6f}",
        "Absolute Difference": "{:.12f}"
    })
)

print("N5 Manual vs scikit-learn Comparison Passed!")


,Profile,Manual N3,scikit-learn N5,Absolute Difference,Match
0,Profile A,0.793851,0.793851,0.000000000000,Yes
1,Profile B,0.030069,0.030069,0.000000000000,Yes


N5 Manual vs scikit-learn Comparison Passed!



### N5 – Mathematical Calculations

The CategoricalNB model uses the same class priors and Laplace-smoothed feature likelihoods as the manual calculations in N3.

For a class c and feature vector x:

$$
\text{Score}(c)=P(c)\prod_{i=1}^{6}P(x_i\mid c)
$$

The posterior probability is calculated by normalizing the two class scores:

$$
P(S\mid x)=
\frac{\text{Score}(S)}
{\text{Score}(S)+\text{Score}(S^c)}
$$

**1. Profile A**

From N3:

$$
\text{Score}_A(S)=0.0006267000
$$

$$
\text{Score}_A(S^c)=0.0001627433
$$

Therefore:

$$
P(S\mid x_A)=
\frac{0.0006267000}
{0.0006267000+0.0001627433}
$$

$$
\boxed{P(S\mid x_A)=0.793851=79.39\%}
$$

The scikit-learn classifier returns the same probability.

**2. Profile B**

From N3:

$$
\text{Score}_B(S)=0.0017521796
$$

$$
\text{Score}_B(S^c)=0.0565192684
$$

Therefore:

$$
P(S\mid x_B)=
\frac{0.0017521796}
{0.0017521796+0.0565192684}
$$

$$
\boxed{P(S\mid x_B)=0.030069=3.01\%}
$$

The scikit-learn classifier returns the same probability.

**Conclusion**

The manual Naive Bayes calculations and scikit-learn predictions match for both profiles, confirming that the class priors, feature likelihoods, Laplace smoothing, and probability normalization are implemented consistently.


In [139]:

# N5 – Final Verification (R3 and R5)

# R3 – Verify chronological training data
assert train["Date"].dt.year.isin([2010, 2011]).all()

assert len(X_train) == len(train) == 4500

# Verify six features are used
assert X_train.columns.tolist() == features

# Verify inputs are binary
assert X_train.isin([0, 1]).all().all()

# Verify class labels
assert np.array_equal(nb.classes_, np.array([0, 1]))

# Verify model prior probabilities
model_priors = np.exp(nb.class_log_prior_)

assert np.isclose(
    model_priors[0],
    prior_not_S
)

assert np.isclose(
    model_priors[1],
    prior_S
)

# R5 – Verify normalized probabilities
assert np.allclose(
    probabilities.sum(axis=1),
    1.0
)

# Verify manual and model predictions match
for i, name in enumerate(["Profile A", "Profile B"]):

    manual_S = n3_results[name]["P(S|x)"]
    manual_not_S = n3_results[name]["P(S^c|x)"]

    assert np.isclose(
        probabilities[i, strong_idx],
        manual_S,
        atol=1e-10,
        rtol=0
    )

    assert np.isclose(
        probabilities[i, not_strong_idx],
        manual_not_S,
        atol=1e-10,
        rtol=0
    )

print("Training data verification: Passed")
print("Model priors verification: Passed")
print("Binary feature verification: Passed")
print("Probability normalization: Passed")
print("Manual vs scikit-learn comparison: Passed")

print("\nN5 Verification Passed! R3 and R5 satisfied.")


Training data verification: Passed
Model priors verification: Passed
Binary feature verification: Passed
Probability normalization: Passed
Manual vs scikit-learn comparison: Passed

N5 Verification Passed! R3 and R5 satisfied.



### N5 – Verification (R3 and R5)

**Rule R3: Training Data Only**

The CategoricalNB classifier was trained using only the 4,500 store-week observations from 2010–2011.

The 2012 testing data was not used to fit the model.

The target variable S and feature G were calculated using training-only store averages.

**Rule R5: Probability Verification**

The predicted probabilities for both classes must sum to 1.

**Profile A**

$$
P(S\mid x_A)+P(S^c\mid x_A)
$$

$$
=0.793851+0.206149=1
$$

**Profile B**

$$
P(S\mid x_B)+P(S^c\mid x_B)
$$

$$
=0.030069+0.969931=1
$$

The predicted probabilities were also independently compared with the manual Naive Bayes calculations from N3.

Both profiles produced matching results within numerical precision.

Therefore, the N5 model satisfies Rules R3 and R5.



### N5 – Interpretation

The CategoricalNB classifier predicts a 79.39% probability of strong sales for Profile A and a 3.01% probability for Profile B.

These predictions match the manual Naive Bayes calculations from N3, confirming the correctness of the model implementation.

Using a 50% classification threshold, Profile A is predicted to be strong, while Profile B is predicted to be non-strong.



### N6 – Naive Bayes Model Evaluation

In this task, we evaluate the trained Naive Bayes classifier on the unseen 2012 testing dataset.

The model was trained using only the 2010–2011 data, following Rule R3.

We compare three prediction methods:

1. **Naive Bayes:** The trained CategoricalNB classifier.
2. **Never Strong Baseline:** Predicts every week as non-strong (S = 0).
3. **Holiday Rule Baseline:** Predicts a strong week (S = 1) only when Holiday_Flag = 1.

We evaluate the methods using:

**Confusion Matrix**

- TP (True Positive): Strong week correctly predicted as strong.
- TN (True Negative): Non-strong week correctly predicted as non-strong.
- FP (False Positive): Non-strong week incorrectly predicted as strong.
- FN (False Negative): Strong week incorrectly predicted as non-strong.

**Accuracy**

$$
\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}
$$

**Precision**

$$
\text{Precision}=\frac{TP}{TP+FP}
$$

**Recall**

$$
\text{Recall}=\frac{TP}{TP+FN}
$$

Precision represents P(S | predicted strong), while recall represents P(predicted strong | S).

If a model makes no positive predictions, precision is undefined. For numerical reporting, we use zero_division=0 and explicitly mention this limitation.

All metrics will be independently verified using confusion-matrix counts, following Rule R5.


In [140]:

# N6 – Prepare Unseen Testing Data

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score
)

# Same six features used during training
features = ["H", "Q", "C", "F", "J", "G"]

# 2012 testing features and actual labels
X_test = test[features].astype(int)
y_test = test["S"].astype(int)

print("Testing feature shape:", X_test.shape)
print("Testing target shape:", y_test.shape)

print("\nActual test class distribution:")
print(y_test.value_counts().sort_index())

print("\nTest date range:")
print(test["Date"].min().date(), "to",
      test["Date"].max().date())

display(X_test.head())


Testing feature shape: (1935, 6)
Testing target shape: (1935,)

Actual test class distribution:
S
0    1680
1     255
Name: count, dtype: int64

Test date range:
2012-01-06 to 2012-10-26


,H,Q,C,F,J,G
100,0,0,0,0,0,1
101,0,0,0,0,0,1
102,0,0,0,0,0,1
103,0,0,0,0,0,1
104,0,0,0,0,0,1


In [141]:

# N6 – Generate Predictions

# 1. Trained Naive Bayes classifier
pred_nb = nb.predict(X_test).astype(int)

# 2. Never-strong baseline
pred_never = np.zeros(len(y_test), dtype=int)

# 3. Holiday-rule baseline
pred_holiday = test["H"].astype(int).to_numpy()

# Store predictions for evaluation
models = {
    "Naive Bayes": pred_nb,
    "Never Strong": pred_never,
    "Holiday Rule": pred_holiday
}

print("Number of positive predictions:")

for name, prediction in models.items():
    print(f"{name}: {int(prediction.sum())}")

print("\nActual strong weeks:", int(y_test.sum()))


Number of positive predictions:
Naive Bayes: 0
Never Strong: 0
Holiday Rule: 90

Actual strong weeks: 255


In [142]:

# N6 – Confusion Matrices and Model Evaluation

n6_results = []

for name, prediction in models.items():

    # Confusion matrix: [[TN, FP], [FN, TP]]
    cm = confusion_matrix(
        y_test,
        prediction,
        labels=[0, 1]
    )

    TN, FP, FN, TP = cm.ravel()

    # Evaluation metrics
    accuracy = accuracy_score(y_test, prediction)

    precision = precision_score(
        y_test,
        prediction,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        prediction,
        zero_division=0
    )

    n6_results.append({
        "Model": name,
        "TN": int(TN),
        "FP": int(FP),
        "FN": int(FN),
        "TP": int(TP),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall
    })

    print(f"\n{name} Confusion Matrix:")
    print(cm)

# Create evaluation table
n6_table = pd.DataFrame(n6_results)

print("\nModel Evaluation Results:")

display(
    n6_table.style.format({
        "Accuracy": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}"
    })
)



Naive Bayes Confusion Matrix:
[[1680    0]
 [ 255    0]]

Never Strong Confusion Matrix:
[[1680    0]
 [ 255    0]]

Holiday Rule Confusion Matrix:
[[1613   67]
 [ 232   23]]

Model Evaluation Results:


,Model,TN,FP,FN,TP,Accuracy,Precision,Recall
0,Naive Bayes,1680,0,255,0,0.868217,0.000000,0.000000
1,Never Strong,1680,0,255,0,0.868217,0.000000,0.000000
2,Holiday Rule,1613,67,232,23,0.845478,0.255556,0.090196


In [143]:

# N6 – Check Holiday Distribution in the Test Set

# Count holiday and November–December observations
test_holiday_count = int(test["H"].sum())
test_season_count = int(test["Q"].sum())

# Check holiday dates appearing in 2012 test data
holiday_test_dates = (
    test.loc[test["H"], "Date"]
    .drop_duplicates()
    .sort_values()
)

print("Holiday store-weeks in 2012:", test_holiday_count)
print("November–December store-weeks:", test_season_count)

print("\nHoliday dates in the test set:")
for date in holiday_test_dates:
    print(date.strftime("%Y-%m-%d"))

# Strong holiday weeks
strong_test_holidays = int(
    (test["S"] & test["H"]).sum()
)

print("\nStrong holiday store-weeks:", strong_test_holidays)

# Verify test-set season and date range
assert test["Date"].dt.year.eq(2012).all()
assert test["Date"].dt.month.max() == 10
assert test_season_count == 0
assert test_holiday_count == 90
assert strong_test_holidays == 23

print("\nHoliday Distribution Verification Passed!")


Holiday store-weeks in 2012: 90
November–December store-weeks: 0

Holiday dates in the test set:
2012-02-10
2012-09-07

Strong holiday store-weeks: 23

Holiday Distribution Verification Passed!



### N6 – Mathematical Calculations

The testing dataset contains:

$$
N=1935
$$

$$
\text{Actual Strong Weeks}=255
$$

$$
\text{Actual Non-Strong Weeks}=1680
$$

#### 1. Naive Bayes Classifier

From the confusion matrix:

- TP = 0
- TN = 1680
- FP = 0
- FN = 255

**Accuracy**

$$
\text{Accuracy}=\frac{TP+TN}{N}
$$

$$
=\frac{0+1680}{1935}
$$

$$
\boxed{\text{Accuracy}=0.868217=86.82\%}
$$

**Precision**

$$
\text{Precision}=\frac{TP}{TP+FP}
=\frac{0}{0+0}
$$

The precision is mathematically undefined because the model predicts no strong weeks.

For numerical evaluation, we use zero_division=0 and report:

$$
\boxed{\text{Precision}=0.00\%}
$$

**Recall**

$$
\text{Recall}=\frac{TP}{TP+FN}
$$

$$
=\frac{0}{0+255}
$$

$$
\boxed{\text{Recall}=0.00\%}
$$

The Naive Bayes classifier misses all 255 actual strong weeks.

---

#### 2. Never Strong Baseline

This baseline predicts every week as non-strong.

- TP = 0
- TN = 1680
- FP = 0
- FN = 255

**Accuracy**

$$
\text{Accuracy}=\frac{1680}{1935}
$$

$$
\boxed{\text{Accuracy}=86.82\%}
$$

**Precision**

Since no strong predictions are made, precision is undefined. It is reported as 0 using zero_division=0.

$$
\boxed{\text{Precision}=0.00\%}
$$

**Recall**

$$
\text{Recall}=\frac{0}{255}
$$

$$
\boxed{\text{Recall}=0.00\%}
$$

---

#### 3. Holiday Rule Baseline

This baseline predicts strong sales whenever H = 1.

From the confusion matrix:

- TP = 23
- TN = 1613
- FP = 67
- FN = 232

**Accuracy**

$$
\text{Accuracy}=\frac{TP+TN}{N}
$$

$$
=\frac{23+1613}{1935}
$$

$$
\boxed{\text{Accuracy}=0.845478=84.55\%}
$$

**Precision**

$$
\text{Precision}=\frac{TP}{TP+FP}
$$

$$
=\frac{23}{23+67}
$$

$$
\boxed{\text{Precision}=0.255556=25.56\%}
$$

**Recall**

$$
\text{Recall}=\frac{TP}{TP+FN}
$$

$$
=\frac{23}{23+232}
$$

$$
\boxed{\text{Recall}=0.090196=9.02\%}
$$

**Fraction of Strong Weeks Missed**

$$
\text{Missed Fraction}=1-\text{Recall}
$$

$$
=\frac{232}{255}
$$

$$
\boxed{\text{Missed Fraction}=90.98\%}
$$

---

#### 4. Model Comparison

| Method | Accuracy | Precision | Recall |
|---|---:|---:|---:|
| Naive Bayes | 86.82% | 0.00% | 0.00% |
| Never Strong | 86.82% | 0.00% | 0.00% |
| Holiday Rule | 84.55% | 25.56% | 9.02% |

Although Naive Bayes achieves 86.82% accuracy, it performs identically to the never-strong baseline at the default classification threshold.

The holiday rule has slightly lower accuracy but successfully identifies 23 of the 255 strong weeks.

Therefore, the Naive Bayes classifier does not outperform the baselines at identifying strong weeks using the default threshold.

---

#### 5. Effect of Missing Thanksgiving and Christmas

The 2012 test dataset ends in October, so it contains no Thanksgiving or Christmas weeks and no November–December observations.

The holiday rule is evaluated only on Super Bowl and Labor Day weeks.

This limits the evaluation because the model is not tested on the Thanksgiving and Christmas sales patterns observed during training.

Therefore, the test results may not represent performance across a complete calendar year.


In [144]:

# N6 – Final Verification (R3 and R5)

# R3 – Verify chronological data split
assert train["Date"].dt.year.isin([2010, 2011]).all()
assert test["Date"].dt.year.eq(2012).all()

assert train["Date"].max() < test["Date"].min()

assert len(test) == 1935
assert int(y_test.sum()) == 255

# Verify test features and labels
assert X_test.columns.tolist() == features
assert X_test.isin([0, 1]).all().all()

# Check Naive Bayes predicted probabilities
test_probabilities = nb.predict_proba(X_test)

assert np.allclose(
    test_probabilities.sum(axis=1),
    1.0
)

# Independently verify metrics for all three methods
for name, prediction in models.items():

    row = n6_table.loc[
        n6_table["Model"] == name
    ].iloc[0]

    TP = int(row["TP"])
    TN = int(row["TN"])
    FP = int(row["FP"])
    FN = int(row["FN"])

    # Confusion matrix count verification
    assert TP + TN + FP + FN == len(y_test)
    assert TP + FN == int(y_test.sum())
    assert TN + FP == int((y_test == 0).sum())

    # Alternative accuracy calculation
    accuracy_check = 1 - (FP + FN) / len(y_test)

    # Precision by counting, with zero-division convention
    precision_check = (
        TP / (TP + FP) if (TP + FP) > 0 else 0.0
    )

    # Recall by counting
    recall_check = TP / (TP + FN)

    assert np.isclose(row["Accuracy"], accuracy_check)
    assert np.isclose(row["Precision"], precision_check)
    assert np.isclose(row["Recall"], recall_check)

    # Recall + missed fraction = 1
    assert np.isclose(
        recall_check + FN / (TP + FN),
        1.0
    )

    print(f"{name}: All metric checks passed.")

# Verify never-strong prediction behavior
assert np.all(pred_never == 0)

# Verify holiday baseline follows the event definition
assert np.array_equal(
    pred_holiday,
    test["H"].astype(int).to_numpy()
)

print("\nTraining/Test Separation: Passed")
print("Probability Normalization: Passed")
print("Confusion Matrix Verification: Passed")
print("Accuracy, Precision, Recall Verification: Passed")

print("\nN6 Verification Passed! R3 and R5 satisfied.")


Naive Bayes: All metric checks passed.
Never Strong: All metric checks passed.
Holiday Rule: All metric checks passed.

Training/Test Separation: Passed
Probability Normalization: Passed
Confusion Matrix Verification: Passed
Accuracy, Precision, Recall Verification: Passed

N6 Verification Passed! R3 and R5 satisfied.



### N6 – Verification (R3 and R5)

**Rule R3: Unseen Testing Data**

The Naive Bayes classifier was trained on 2010–2011 data and evaluated on the unseen 2012 dataset containing 1,935 store-weeks.

Store averages for S and G were calculated using training data only.

**Rule R5: Independent Verification**

For each confusion matrix:

$$
TP+TN+FP+FN=1935
$$

For example, the holiday rule satisfies:

$$
23+1613+67+232=1935
$$

Accuracy is independently verified using:

$$
\text{Accuracy}=1-\frac{FP+FN}{N}
$$

For the holiday rule:

$$
1-\frac{67+232}{1935}=0.845478
$$

Recall is verified using its complement, the missed fraction:

$$
\text{Recall}+\text{Missed Fraction}=1
$$

For the holiday rule:

$$
\frac{23}{255}+\frac{232}{255}=1
$$

For the Naive Bayes and never-strong models, no positive predictions are made, so precision is undefined and is reported as 0 using the chosen evaluation convention.

The Naive Bayes predicted class probabilities also sum to 1 for every test observation.

All verification checks passed.



### N6 – Interpretation

The Naive Bayes classifier achieves 86.82% accuracy but has 0% recall because it predicts every test week as non-strong. Its performance at the default threshold is therefore identical to the never-strong baseline.

The holiday rule has lower accuracy (84.55%) but achieves 25.56% precision and 9.02% recall, correctly identifying 23 strong weeks.

Thus, Naive Bayes does not improve the identification of strong sales weeks at the default threshold. Accuracy alone is misleading because most testing weeks are non-strong.

Additionally, the 2012 dataset ends in October and contains no Thanksgiving or Christmas weeks. This limits our ability to evaluate model performance during the November–December holiday season.



### N7 – Threshold Analysis

In this task, we investigate how changing the classification threshold affects the precision and recall of our Naive Bayes classifier.

The model estimates the probability of a strong sales week:

$$
P(S\mid x)
$$

We predict a strong week when:

$$
P(S\mid x)>t
$$

where t is the classification threshold.

We evaluate four thresholds:

$$
t\in\{0.1,0.2,0.3,0.5\}
$$

**Precision**

Precision measures the proportion of predicted strong weeks that are actually strong.

$$
\text{Precision}=\frac{TP}{TP+FP}
$$

**Recall**

Recall measures the proportion of actual strong weeks correctly identified.

$$
\text{Recall}=\frac{TP}{TP+FN}
$$

Lower thresholds generally produce more positive predictions, potentially improving recall but increasing false positives.

We will evaluate all four thresholds on the unseen 2012 testing dataset and independently verify the resulting metrics, following R3 and R5.

When no positive predictions are made, precision is mathematically undefined and is reported as 0 using zero_division=0.


In [145]:

# N7 – Obtain Strong-Week Probabilities

# Use the existing N5 Naive Bayes model
# and the 2012 test set from N6

features = ["H", "Q", "C", "F", "J", "G"]

X_test = test[features].astype(int)
y_test = test["S"].astype(int)

# Get probabilities for both classes
test_probabilities = nb.predict_proba(X_test)

# Find the column corresponding to strong weeks (S=1)
strong_class_idx = list(nb.classes_).index(1)

# Extract P(S|x)
p_strong_test = test_probabilities[:, strong_class_idx]

print("Number of test observations:", len(p_strong_test))

print("Minimum predicted probability:",
      round(p_strong_test.min(), 6))

print("Maximum predicted probability:",
      round(p_strong_test.max(), 6))

print("\nFirst five predicted strong-week probabilities:")
print(np.round(p_strong_test[:5], 6))


Number of test observations: 1935
Minimum predicted probability: 0.024723
Maximum predicted probability: 0.316865

First five predicted strong-week probabilities:
[0.054333 0.054333 0.054333 0.054333 0.054333]


In [146]:

# N7 – Precision and Recall for Different Thresholds

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score
)

# Required thresholds
thresholds = [0.1, 0.2, 0.3, 0.5]

n7_results = []

for t in thresholds:

    # Predict strong when P(S|x) > threshold
    predictions = (p_strong_test > t).astype(int)

    # Confusion matrix
    cm = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    )

    TN, FP, FN, TP = cm.ravel()

    # Evaluation metrics
    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    n7_results.append({
        "Threshold": t,
        "Predicted Strong": int(predictions.sum()),
        "TP": int(TP),
        "FP": int(FP),
        "FN": int(FN),
        "TN": int(TN),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall
    })

n7_table = pd.DataFrame(n7_results)

display(
    n7_table.style.format({
        "Threshold": "{:.1f}",
        "Accuracy": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}"
    })
)


,Threshold,Predicted Strong,TP,FP,FN,TN,Accuracy,Precision,Recall
0,0.1,125,19,106,236,1574,0.823256,0.152000,0.074510
1,0.2,10,3,7,252,1673,0.866150,0.300000,0.011765
2,0.3,2,0,2,255,1678,0.867183,0.000000,0.000000
3,0.5,0,0,0,255,1680,0.868217,0.000000,0.000000



### N7 – Mathematical Calculations

The test dataset contains 1,935 store-weeks:

- Actual strong weeks: 255
- Actual non-strong weeks: 1,680

For each threshold, the model predicts a strong week when:

$$
P(S\mid x)>t
$$

**1. Threshold t = 0.1**

Confusion matrix counts:

- TP = 19
- FP = 106
- FN = 236
- TN = 1574

Precision:

$$
\text{Precision}=\frac{19}{19+106}
$$

$$
\boxed{\text{Precision}=0.1520=15.20\%}
$$

Recall:

$$
\text{Recall}=\frac{19}{19+236}
$$

$$
\boxed{\text{Recall}=0.0745=7.45\%}
$$

**2. Threshold t = 0.2**

Confusion matrix counts:

- TP = 3
- FP = 7
- FN = 252
- TN = 1673

Precision:

$$
\text{Precision}=\frac{3}{3+7}
$$

$$
\boxed{\text{Precision}=0.3000=30.00\%}
$$

Recall:

$$
\text{Recall}=\frac{3}{3+252}
$$

$$
\boxed{\text{Recall}=0.0118=1.18\%}
$$

**3. Threshold t = 0.3**

Confusion matrix counts:

- TP = 0
- FP = 2
- FN = 255
- TN = 1678

Precision:

$$
\text{Precision}=\frac{0}{0+2}=0
$$

$$
\boxed{\text{Precision}=0.00\%}
$$

Recall:

$$
\text{Recall}=\frac{0}{0+255}=0
$$

$$
\boxed{\text{Recall}=0.00\%}
$$

**4. Threshold t = 0.5**

Confusion matrix counts:

- TP = 0
- FP = 0
- FN = 255
- TN = 1680

Precision:

$$
\text{Precision}=\frac{0}{0+0}
$$

Precision is undefined because no positive predictions are made. Using zero_division=0, we report:

$$
\boxed{\text{Precision}=0.00\%}
$$

Recall:

$$
\text{Recall}=\frac{0}{0+255}=0
$$

$$
\boxed{\text{Recall}=0.00\%}
$$

**Threshold Comparison**

| Threshold | Precision | Recall |
|---|---:|---:|
| 0.1 | 15.20% | 7.45% |
| 0.2 | 30.00% | 1.18% |
| 0.3 | 0.00% | 0.00% |
| 0.5 | 0.00% | 0.00% |

Lowering the threshold from 0.5 to 0.1 increases recall from 0% to 7.45%, allowing the model to identify some strong weeks. However, the model also produces more false positives.

Threshold 0.2 has higher precision than 0.1, but much lower recall.


In [147]:

# N7 – Independent Verification (R3 and R5)

# R3: Verify test data is from 2012
assert test["Date"].dt.year.eq(2012).all()
assert train["Date"].dt.year.isin([2010, 2011]).all()

assert len(y_test) == 1935
assert int(y_test.sum()) == 255

# R5: Verify predicted class probabilities
assert np.allclose(
    test_probabilities.sum(axis=1),
    1.0
)

previous_positive_count = len(y_test) + 1
previous_recall = 1.0

for t in thresholds:

    # Recalculate predictions directly
    prediction = p_strong_test > t
    actual = y_test.to_numpy().astype(bool)

    # Independently count TP, FP, FN, TN
    TP = int(np.sum(actual & prediction))
    FP = int(np.sum(~actual & prediction))
    FN = int(np.sum(actual & ~prediction))
    TN = int(np.sum(~actual & ~prediction))

    # Find stored results
    row = n7_table.loc[
        np.isclose(n7_table["Threshold"], t)
    ].iloc[0]

    # Verify confusion matrix counts
    assert TP == row["TP"]
    assert FP == row["FP"]
    assert FN == row["FN"]
    assert TN == row["TN"]

    assert TP + FP + FN + TN == len(y_test)
    assert TP + FN == 255
    assert TN + FP == 1680

    # Independently calculate metrics
    precision_check = (
        TP / (TP + FP) if TP + FP > 0 else 0.0
    )

    recall_check = TP / (TP + FN)

    accuracy_check = (TP + TN) / len(y_test)

    # Verify stored metrics
    assert np.isclose(
        row["Precision"], precision_check
    )

    assert np.isclose(
        row["Recall"], recall_check
    )

    assert np.isclose(
        row["Accuracy"], accuracy_check
    )

    # R5: Verify recall and missed fraction
    missed_fraction = FN / (TP + FN)

    assert np.isclose(
        recall_check + missed_fraction,
        1.0
    )

    # Increasing threshold cannot increase positive count
    positive_count = TP + FP

    assert positive_count <= previous_positive_count
    assert recall_check <= previous_recall + 1e-12

    previous_positive_count = positive_count
    previous_recall = recall_check

    print(f"Threshold {t:.1f}: All checks passed.")

# Verify t=0.5 matches the default N6 classifier
pred_05 = (p_strong_test > 0.5).astype(int)

assert np.array_equal(pred_05, pred_nb)

print("\nTraining/Test Separation: Passed")
print("Probability Normalization: Passed")
print("Confusion Matrix Verification: Passed")
print("Precision and Recall Verification: Passed")
print("Default Threshold Comparison: Passed")

print("\nN7 Verification Passed! R3 and R5 satisfied.")


Threshold 0.1: All checks passed.
Threshold 0.2: All checks passed.
Threshold 0.3: All checks passed.
Threshold 0.5: All checks passed.

Training/Test Separation: Passed
Probability Normalization: Passed
Confusion Matrix Verification: Passed
Precision and Recall Verification: Passed
Default Threshold Comparison: Passed

N7 Verification Passed! R3 and R5 satisfied.



### N7 – Verification (R3 and R5)

**Rule R3: Unseen Testing Data**

The threshold analysis uses the trained Naive Bayes model from N5 and the unseen 2012 testing dataset.

The model was trained on 2010–2011 observations only.

**Rule R5: Independent Probability Verification**

Each confusion matrix must satisfy:

$$
TP+FP+FN+TN=1935
$$

For threshold 0.1:

$$
19+106+236+1574=1935
$$

For threshold 0.2:

$$
3+7+252+1673=1935
$$

For threshold 0.3:

$$
0+2+255+1678=1935
$$

For threshold 0.5:

$$
0+0+255+1680=1935
$$

Recall is verified using its complement, the missed fraction:

$$
\text{Recall}+\text{Missed Fraction}=1
$$

For threshold 0.1:

$$
\frac{19}{255}+\frac{236}{255}=1
$$

For threshold 0.2:

$$
\frac{3}{255}+\frac{252}{255}=1
$$

For thresholds 0.3 and 0.5:

$$
0+\frac{255}{255}=1
$$

The Python code independently verifies the confusion-matrix counts, accuracy, precision, recall, and probability normalization for all four thresholds.

All checks passed, satisfying Rules R3 and R5.



### N7 – Interpretation

At threshold 0.1, the Naive Bayes model achieves 15.20% precision and 7.45% recall, identifying 19 strong weeks. At threshold 0.2, precision increases to 30.00%, but recall decreases to 1.18%, identifying only 3 strong weeks.

At thresholds 0.3 and 0.5, recall is 0% because the model fails to identify any strong weeks.

These results demonstrate the trade-off between precision and recall. Lower thresholds identify more strong weeks but also increase false positives.

Although lowering the threshold improves recall compared with the default threshold, the model still misses most strong weeks. At threshold 0.1, its recall of 7.45% remains below the holiday baseline's recall of 9.02%.
